# AIXL cross-family judges (Colab, GPU)
Three open models of different families act as independent arbiters over 630 synthetic instruction pairs (no private data). Runtime > Change runtime type > **T4 GPU**, then Run all. Results print at the end as `id=letter` lists (S = SAME, D = DIFFERENT, U = UNSURE, I = invalid).

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
RULES = "You are an arbiter. Each input line is {\"id\",\"a\",\"b\"}: two instructions (Spanish, English or Portuguese) that will be given to an AI system. Decide whether they ask for EXACTLY the same thing.\n- SAME: same action, same objects, same recipients, numbers, units, times, frequencies, channels, scope, conditions and polarity. Different words, synonyms (delete/remove, install/deploy), different order, translation between languages and number words vs digits are all fine.\n- DIFFERENT: any detail added, dropped or changed (even one: a recipient, a number, a day, a channel, a scope word like all/only/except, a condition, a deadline direction), or opposite instructions.\n- UNSURE: only if you truly cannot tell from the two texts.\nBe strict: when one text contains a detail the other does not state, answer DIFFERENT.\nOutput exactly one line per input, `id<TAB>verdict<TAB>short reason`, same ids, nothing else.\n"
DATA = "{\"id\": \"J001\", \"a\": \"Voc\u00ea s\u00f3 pode editar o documento se for o autor.\", \"b\": \"Voc\u00ea pode editar o documento, se for o autor.\"}\n{\"id\": \"J002\", \"a\": \"Cada equipo debe tener como m\u00e1ximo cinco personas.\", \"b\": \"Ning\u00fan equipo puede superar las cinco personas.\"}\n{\"id\": \"J003\", \"a\": \"Each team needs at least 3 engineers and at most 5 interns.\", \"b\": \"Each team needs at most 3 engineers and at least 5 interns.\"}\n{\"id\": \"J004\", \"a\": \"Send the draft to legal on Tuesday and publish it on Thursday.\", \"b\": \"Send the draft to legal and publish it on Thursday.\"}\n{\"id\": \"J005\", \"a\": \"Can I approve refunds myself? Yes, up to 50 dollars per month.\", \"b\": \"Can I approve refunds myself? Yes, up to 50 dollars per ticket.\"}\n{\"id\": \"J006\", \"a\": \"N\u00e3o aprove reembolsos que excedam 500 reais sem a assinatura do gerente.\", \"b\": \"Aprove reembolsos que n\u00e3o excedam 500 reais sem a assinatura do gerente.\"}\n{\"id\": \"J007\", \"a\": \"S\u00f3 emita o certificado se o aluno tiver frequ\u00eancia m\u00ednima de 75%.\", \"b\": \"Sem frequ\u00eancia de pelo menos 75%, o certificado n\u00e3o deve ser emitido.\"}\n{\"id\": \"J008\", \"a\": \"Salvo los cajeros, el personal debe cerrar la caja antes de salir.\", \"b\": \"El personal debe cerrar la caja antes de salir, excepto los cajeros.\"}\n{\"id\": \"J009\", \"a\": \"N\u00e3o \u00e9 obrigat\u00f3rio usar m\u00e1scara, mas \u00e9 proibido entrar sem higienizar as m\u00e3os.\", \"b\": \"Higienizar as m\u00e3os antes de entrar \u00e9 obrigat\u00f3rio; a m\u00e1scara \u00e9 facultativa.\"}\n{\"id\": \"J010\", \"a\": \"O gerente deve aprovar e pode assinar o contrato.\", \"b\": \"O gerente pode aprovar e deve assinar o contrato.\"}\n{\"id\": \"J011\", \"a\": \"O sistema envia o alerta ao gerente, que aprova o pedido.\", \"b\": \"O gerente envia o alerta ao sistema, que aprova o pedido.\"}\n{\"id\": \"J012\", \"a\": \"O financeiro aprova o pagamento ao fornecedor ap\u00f3s a entrega.\", \"b\": \"O fornecedor aprova o pagamento ao financeiro ap\u00f3s a entrega.\"}\n{\"id\": \"J013\", \"a\": \"\u00bfCobro el env\u00edo? No, a menos que pese m\u00e1s de 5 kg.\", \"b\": \"\u00bfCobro el env\u00edo? No, ni siquiera si pesa m\u00e1s de 5 kg.\"}\n{\"id\": \"J014\", \"a\": \"Devo ligar para o cliente? N\u00e3o, envie um e-mail.\", \"b\": \"Devo ligar para o cliente? N\u00e3o. Mande um e-mail.\"}\n{\"id\": \"J015\", \"a\": \"A equipe de suporte resolve o chamado e registra a solu\u00e7\u00e3o.\", \"b\": \"O chamado \u00e9 resolvido pela equipe de suporte, que registra a solu\u00e7\u00e3o.\"}\n{\"id\": \"J016\", \"a\": \"Os visitantes podem usar o wi-fi, mas precisam de senha.\", \"b\": \"Para usar o wi-fi, os visitantes precisam de senha, e isso \u00e9 permitido.\"}\n{\"id\": \"J017\", \"a\": \"Restart the service only if the health check fails.\", \"b\": \"Restart the service if the health check fails.\"}\n{\"id\": \"J018\", \"a\": \"Unless it is raining, the drill takes place outdoors.\", \"b\": \"The drill takes place outdoors, except when it is raining.\"}\n{\"id\": \"J019\", \"a\": \"No se debe compartir la contrase\u00f1a de administrador con personal externo.\", \"b\": \"El personal externo no debe recibir la contrase\u00f1a de administrador compartida.\"}\n{\"id\": \"J020\", \"a\": \"Se necesitan al menos 3 firmas para pagos de m\u00e1s de 10.000 d\u00f3lares.\", \"b\": \"Se necesitan m\u00e1s de 3 firmas para pagos de al menos 10.000 d\u00f3lares.\"}\n{\"id\": \"J021\", \"a\": \"Escale el incidente si el cliente es premium y el fallo dura m\u00e1s de una hora.\", \"b\": \"Escale el incidente si el cliente es premium o el fallo dura m\u00e1s de una hora.\"}\n{\"id\": \"J022\", \"a\": \"Every employee except interns may request remote work after the probation period.\", \"b\": \"Interns aside, all employees can ask to work remotely once their probation is over.\"}\n{\"id\": \"J023\", \"a\": \"Submit the form no later than the last working day of the month.\", \"b\": \"The form must reach us by the final business day of each month at the latest.\"}\n{\"id\": \"J024\", \"a\": \"Posso abrir o port\u00e3o? N\u00e3o, abra a janela.\", \"b\": \"Posso abrir o port\u00e3o? N\u00e3o abra a janela.\"}\n{\"id\": \"J025\", \"a\": \"Refunds are issued only if the item was returned unused.\", \"b\": \"Refunds are issued if the item was returned unused.\"}\n{\"id\": \"J026\", \"a\": \"Revise los permisos antes del jueves y archive los registros despu\u00e9s del viernes.\", \"b\": \"Revise los permisos despu\u00e9s del viernes y archive los registros antes del jueves.\"}\n{\"id\": \"J027\", \"a\": \"The cashier must not accept cards that are expired, or checks that are post-dated.\", \"b\": \"The cashier must not accept cards that are expired, or checks that are not post-dated.\"}\n{\"id\": \"J028\", \"a\": \"O fornecedor deve pagar a multa ao comprador.\", \"b\": \"O comprador deve pagar a multa ao fornecedor.\"}\n{\"id\": \"J029\", \"a\": \"Descongele a carne na v\u00e9spera e tempere-a uma hora antes de assar.\", \"b\": \"Tempere a carne uma hora antes de lev\u00e1-la ao forno, depois de descongel\u00e1-la no dia anterior.\"}\n{\"id\": \"J030\", \"a\": \"Aprueba los gastos de m\u00e1s de 200 euros.\", \"b\": \"Aprueba los gastos de al menos 200 euros.\"}\n{\"id\": \"J031\", \"a\": \"Do not merge the branch until all tests pass.\", \"b\": \"The branch may be merged only after every test has passed.\"}\n{\"id\": \"J032\", \"a\": \"Ningu\u00e9m pode alterar a tabela de pre\u00e7os sem aprova\u00e7\u00e3o do financeiro.\", \"b\": \"A tabela de pre\u00e7os s\u00f3 pode ser alterada com aprova\u00e7\u00e3o do financeiro.\"}\n{\"id\": \"J033\", \"a\": \"Refund every order except ones shipped after Monday.\", \"b\": \"Refund every order except ones shipped before Monday.\"}\n{\"id\": \"J034\", \"a\": \"S\u00f3 reembolse o pedido se o produto estiver lacrado.\", \"b\": \"O reembolso s\u00f3 \u00e9 permitido quando o produto est\u00e1 lacrado.\"}\n{\"id\": \"J035\", \"a\": \"A n\u00e3o ser que o cliente pe\u00e7a, n\u00e3o envie fatura impressa.\", \"b\": \"Se o cliente pedir, n\u00e3o envie fatura impressa.\"}\n{\"id\": \"J036\", \"a\": \"Puede publicarse cualquier informe, salvo los financieros que sean internos.\", \"b\": \"Puede publicarse cualquier informe, salvo los financieros.\"}\n{\"id\": \"J037\", \"a\": \"If the alarm sounds, evacuate and call security.\", \"b\": \"If the alarm sounds, evacuate; call security if anyone is missing.\"}\n{\"id\": \"J038\", \"a\": \"Fa\u00e7a o backup \u00e0s 22h e reinicie o servidor \u00e0s 23h.\", \"b\": \"Fa\u00e7a o backup \u00e0s 23h e reinicie o servidor \u00e0s 22h.\"}\n{\"id\": \"J039\", \"a\": \"El t\u00e9cnico debe revisar el equipo y puede reemplazar la bater\u00eda.\", \"b\": \"El t\u00e9cnico puede revisar el equipo y debe reemplazar la bater\u00eda.\"}\n{\"id\": \"J040\", \"a\": \"Use a minimum of two eggs and no more than 100 g of sugar.\", \"b\": \"Sugar must not exceed 100 g, and you need two or more eggs.\"}\n{\"id\": \"J041\", \"a\": \"Except for the reporting server, all servers must be patched on Mondays.\", \"b\": \"Except for the reporting server and the backup server, all servers must be patched on Mondays.\"}\n{\"id\": \"J042\", \"a\": \"Pedidos com menos de 3 itens pagam frete.\", \"b\": \"Pedidos com at\u00e9 3 itens pagam frete.\"}\n{\"id\": \"J043\", \"a\": \"Salvo el gerente de turno, nadie puede abrir la caja fuerte.\", \"b\": \"\u00danicamente el gerente de turno puede abrir la caja fuerte.\"}\n{\"id\": \"J044\", \"a\": \"HR sends the offer letter and the candidate signs it.\", \"b\": \"The candidate signs the offer letter that HR sends.\"}\n{\"id\": \"J045\", \"a\": \"Do not share the password with anyone, including your manager.\", \"b\": \"Never share the password with anybody, your manager included.\"}\n{\"id\": \"J046\", \"a\": \"Archive todos los reportes excepto los borradores confidenciales.\", \"b\": \"Archive todos los reportes excepto los borradores y los confidenciales.\"}\n{\"id\": \"J047\", \"a\": \"El t\u00e9cnico debe apagar el equipo y puede avisar al supervisor.\", \"b\": \"El t\u00e9cnico puede apagar el equipo y debe avisar al supervisor.\"}\n{\"id\": \"J048\", \"a\": \"A equipe deve testar o patch antes de aplic\u00e1-lo e pode adiar o rein\u00edcio.\", \"b\": \"A equipe pode testar o patch antes de aplic\u00e1-lo e deve adiar o rein\u00edcio.\"}\n{\"id\": \"J049\", \"a\": \"If the patient is under sixteen, a guardian must consent.\", \"b\": \"A guardian must consent whenever the patient is under sixteen.\"}\n{\"id\": \"J050\", \"a\": \"Cada equipo debe tener al menos 3 ingenieros.\", \"b\": \"Cada equipo debe contar con 3 ingenieros o m\u00e1s.\"}\n{\"id\": \"J051\", \"a\": \"\u00c0s 8h abra o caixa e \u00e0s 18h feche-o.\", \"b\": \"O caixa deve ser aberto \u00e0s 8h e fechado \u00e0s 18h.\"}\n{\"id\": \"J052\", \"a\": \"Todos los vuelos se reembolsan excepto los de tarifa b\u00e1sica comprados con millas.\", \"b\": \"Todos los vuelos se reembolsan excepto los de tarifa b\u00e1sica y los comprados con millas.\"}\n{\"id\": \"J053\", \"a\": \"You can skip the optional survey, but the safety quiz must be completed.\", \"b\": \"Completing the safety quiz is mandatory; the survey is optional and may be skipped.\"}\n{\"id\": \"J054\", \"a\": \"N\u00e3o \u00e9 permitido liberar o pagamento antes da aprova\u00e7\u00e3o do gerente, mas o pedido pode ser cadastrado antes.\", \"b\": \"\u00c9 permitido liberar o pagamento antes da aprova\u00e7\u00e3o do gerente, mas o pedido n\u00e3o pode ser cadastrado antes.\"}\n{\"id\": \"J055\", \"a\": \"The manager must approve the request before HR processes it.\", \"b\": \"HR must approve the request before the manager processes it.\"}\n{\"id\": \"J056\", \"a\": \"El pago se realiza el 5 de cada mes; el reporte, el 1 de cada mes.\", \"b\": \"El pago se realiza el 1 de cada mes; el reporte, el 5 de cada mes.\"}\n{\"id\": \"J057\", \"a\": \"El m\u00e9dico notifica al paciente la cita; la enfermera la confirma.\", \"b\": \"El m\u00e9dico confirma la cita al paciente; la enfermera la notifica.\"}\n{\"id\": \"J058\", \"a\": \"Cuando el servidor falle, rein\u00edcielo y avise.\", \"b\": \"Cuando el servidor falle dos veces, rein\u00edcielo y avise.\"}\n{\"id\": \"J059\", \"a\": \"Can I skip the review? No, merge after two approvals.\", \"b\": \"Can I skip the review? Yes, merge after two approvals.\"}\n{\"id\": \"J060\", \"a\": \"All visitors must sign in except the contractors who have a standing badge.\", \"b\": \"All visitors must sign in except the contractors, who have a standing badge.\"}\n{\"id\": \"J061\", \"a\": \"Posso ausentar-me sem aviso? N\u00e3o, exceto em caso m\u00e9dico com comprovante.\", \"b\": \"Posso ausentar-me sem aviso? N\u00e3o, exceto em caso m\u00e9dico, sem comprovante.\"}\n{\"id\": \"J062\", \"a\": \"O gerente deve aprovar o or\u00e7amento e pode delegar a revis\u00e3o.\", \"b\": \"O gerente pode aprovar o or\u00e7amento e deve delegar a revis\u00e3o.\"}\n{\"id\": \"J063\", \"a\": \"Do not email customers whose invoices are not overdue.\", \"b\": \"Do not email customers whose invoices are overdue.\"}\n{\"id\": \"J064\", \"a\": \"Posso apagar os logs antigos? Sim, depois de arquiv\u00e1-los.\", \"b\": \"Posso apagar os logs antigos? Sim, em vez de arquiv\u00e1-los.\"}\n{\"id\": \"J065\", \"a\": \"Submit timesheets by Friday noon and expenses by Monday.\", \"b\": \"Expenses are due Monday; timesheets are due by noon on Friday.\"}\n{\"id\": \"J066\", \"a\": \"Que el chef pruebe la salsa antes de que la sirva el camarero.\", \"b\": \"Que el camarero pruebe la salsa antes de que la sirva el chef.\"}\n{\"id\": \"J067\", \"a\": \"Never leave the freezer door open for more than thirty seconds.\", \"b\": \"Do not keep the freezer door open longer than thirty seconds.\"}\n{\"id\": \"J068\", \"a\": \"Should we update the client? Don't, call them tomorrow.\", \"b\": \"Should we update the client? Don't call them tomorrow.\"}\n{\"id\": \"J069\", \"a\": \"If the load exceeds 80 percent, add a node; otherwise leave the cluster as is.\", \"b\": \"Add a node when the load is above 80 percent, and leave the cluster unchanged when it is not.\"}\n{\"id\": \"J070\", \"a\": \"Keep at least 3 backups and at most 10 snapshots.\", \"b\": \"Keep at most 3 backups and at least 10 snapshots.\"}\n{\"id\": \"J071\", \"a\": \"El gasto se aprueba siempre que haya presupuesto, aunque falte la factura.\", \"b\": \"El gasto se aprueba siempre que haya presupuesto, salvo que falte la factura.\"}\n{\"id\": \"J072\", \"a\": \"Don't let anyone without a visitor pass past the lobby.\", \"b\": \"Nobody without a visitor pass may get past the lobby.\"}\n{\"id\": \"J073\", \"a\": \"Paga la factura antes del d\u00eda 5 y arch\u00edvala despu\u00e9s del d\u00eda 10.\", \"b\": \"Paga la factura despu\u00e9s del d\u00eda 5 y arch\u00edvala antes del d\u00eda 10.\"}\n{\"id\": \"J074\", \"a\": \"Approve all expense reports except those from contractors.\", \"b\": \"Approve all expense reports except those from contractors and interns.\"}\n{\"id\": \"J075\", \"a\": \"Si llueve, el evento se traslada al sal\u00f3n; si no, se hace en el patio.\", \"b\": \"El evento ser\u00e1 en el patio, a menos que llueva, en cuyo caso ser\u00e1 en el sal\u00f3n.\"}\n{\"id\": \"J076\", \"a\": \"Exceto em caso de emerg\u00eancia, ningu\u00e9m deve sair do laborat\u00f3rio sem avisar.\", \"b\": \"Ningu\u00e9m deve sair do laborat\u00f3rio sem avisar, exceto se for uma emerg\u00eancia m\u00e9dica.\"}\n{\"id\": \"J077\", \"a\": \"Todos os acessos ser\u00e3o revogados, exceto o da administradora.\", \"b\": \"Todos os acessos ser\u00e3o revogados, exceto o dos administradores.\"}\n{\"id\": \"J078\", \"a\": \"El cajero entrega el recibo al cliente.\", \"b\": \"El recibo es entregado por el cajero al cliente.\"}\n{\"id\": \"J079\", \"a\": \"Caso o voo atrase mais de duas horas, ofere\u00e7a um voucher de refei\u00e7\u00e3o.\", \"b\": \"Ofere\u00e7a um voucher de refei\u00e7\u00e3o sempre que o atraso do voo passar de duas horas.\"}\n{\"id\": \"J080\", \"a\": \"Haz el pedido antes de las 10:00 y revisa el inventario por la tarde.\", \"b\": \"Revisa el inventario antes de las 10:00 y haz el pedido por la tarde.\"}\n{\"id\": \"J081\", \"a\": \"Pague la factura antes del 5 de mayo y confirme el pago el 6.\", \"b\": \"El 6 confirme el pago; la factura debe pagarse antes del 5 de mayo.\"}\n{\"id\": \"J082\", \"a\": \"Reembolse todas as despesas, a n\u00e3o ser que n\u00e3o tenham recibo.\", \"b\": \"Reembolse todas as despesas, a n\u00e3o ser que tenham recibo.\"}\n{\"id\": \"J083\", \"a\": \"Notify the client when the payment fails, unless the card is a test card.\", \"b\": \"Notify the client only when the payment fails, unless the card is a test card.\"}\n{\"id\": \"J084\", \"a\": \"N\u00e3o use mais de 2 GB por usu\u00e1rio.\", \"b\": \"Cada usu\u00e1rio pode usar no m\u00e1ximo 2 GB.\"}\n{\"id\": \"J085\", \"a\": \"Aplique o patch na sexta \u00e0 noite e reinicie os servidores no s\u00e1bado.\", \"b\": \"Aplique o patch na sexta e reinicie os servidores no s\u00e1bado \u00e0 noite.\"}\n{\"id\": \"J086\", \"a\": \"Invite no more than 20 guests and order food for at least 25.\", \"b\": \"Invite at least 20 guests and order food for no more than 25.\"}\n{\"id\": \"J087\", \"a\": \"Cada servidor deve ter no m\u00ednimo 2 discos e no m\u00e1ximo 8 gigabytes de RAM.\", \"b\": \"Cada servidor deve ter no m\u00e1ximo 2 discos e no m\u00ednimo 8 gigabytes de RAM.\"}\n{\"id\": \"J088\", \"a\": \"You must lock the screen and may log out when leaving your desk.\", \"b\": \"You may lock the screen and must log out when leaving your desk.\"}\n{\"id\": \"J089\", \"a\": \"Posso reembolsar a despesa sem nota? Sim, mas s\u00f3 at\u00e9 100 reais, com nota.\", \"b\": \"Posso reembolsar a despesa sem nota? Sim, mas s\u00f3 at\u00e9 100 reais, sem nota.\"}\n{\"id\": \"J090\", \"a\": \"Todos os funcion\u00e1rios devem registrar o ponto, exceto os gerentes.\", \"b\": \"Todos os funcion\u00e1rios devem registrar o ponto, exceto os gerentes e os diretores.\"}\n{\"id\": \"J091\", \"a\": \"Don't approve the loan unless the applicant has a credit score of 700 or higher.\", \"b\": \"Approve the loan only if the applicant's credit score is at least 700.\"}\n{\"id\": \"J092\", \"a\": \"Preciso reiniciar o roteador? N\u00e3o, basta desligar o Wi-Fi.\", \"b\": \"N\u00e3o \u00e9 necess\u00e1rio reiniciar o roteador; desligar o Wi-Fi j\u00e1 \u00e9 suficiente.\"}\n{\"id\": \"J093\", \"a\": \"El cliente notificar\u00e1 al proveedor del retraso.\", \"b\": \"El proveedor notificar\u00e1 al cliente del retraso.\"}\n{\"id\": \"J094\", \"a\": \"\u00bfLlamo al proveedor? No, escr\u00edbele hoy.\", \"b\": \"\u00bfLlamo al proveedor? No le escribas hoy.\"}\n{\"id\": \"J095\", \"a\": \"New hires are assigned a mentor by their team lead.\", \"b\": \"The team lead assigns each new hire a mentor.\"}\n{\"id\": \"J096\", \"a\": \"Is overtime allowed on weekends? No, unless a manager signs the request.\", \"b\": \"Is overtime allowed on weekends? No. Only a manager's signed request allows it.\"}\n{\"id\": \"J097\", \"a\": \"Se permiten m\u00e1s de 10 usuarios por licencia y menos de 3 dispositivos.\", \"b\": \"Se permiten menos de 10 usuarios por licencia y m\u00e1s de 3 dispositivos.\"}\n{\"id\": \"J098\", \"a\": \"Envie a fatura na segunda e pague o fornecedor na sexta.\", \"b\": \"O fornecedor deve ser pago na sexta; a fatura, enviada na segunda.\"}\n{\"id\": \"J099\", \"a\": \"Se a senha expirar, o usu\u00e1rio deve redefini-la.\", \"b\": \"O usu\u00e1rio deve redefinir a senha quando ela expirar.\"}\n{\"id\": \"J100\", \"a\": \"Soporte t\u00e9cnico asigna cada ticket al ingeniero de guardia.\", \"b\": \"El ingeniero de guardia recibe de soporte t\u00e9cnico la asignaci\u00f3n de cada ticket.\"}\n{\"id\": \"J101\", \"a\": \"Posso acumular f\u00e9rias? Sim, at\u00e9 30 dias por ano.\", \"b\": \"Posso acumular f\u00e9rias? Sim, desde que n\u00e3o passe de 30 dias por ano.\"}\n{\"id\": \"J102\", \"a\": \"Agents should escalate billing disputes and must close duplicate tickets.\", \"b\": \"Agents must escalate billing disputes and should close duplicate tickets.\"}\n{\"id\": \"J103\", \"a\": \"Other than the lobby, every area requires a badge.\", \"b\": \"A badge is required everywhere except the lobby.\"}\n{\"id\": \"J104\", \"a\": \"Fewer than ten people may join the session.\", \"b\": \"No more than nine people may join the session.\"}\n{\"id\": \"J105\", \"a\": \"O operador deve usar luvas e pode usar \u00f3culos.\", \"b\": \"As luvas s\u00e3o obrigat\u00f3rias para o operador; os \u00f3culos s\u00e3o opcionais.\"}\n{\"id\": \"J106\", \"a\": \"Cuando el sensor detecte humo, la alarma suena, a menos que est\u00e9 en mantenimiento.\", \"b\": \"La alarma suena cada vez que el sensor detecta humo, salvo durante el mantenimiento.\"}\n{\"id\": \"J107\", \"a\": \"No se aceptan m\u00e1s de dos maletas por pasajero.\", \"b\": \"Cada pasajero puede llevar dos maletas como m\u00e1ximo.\"}\n{\"id\": \"J108\", \"a\": \"Os alunos devem entregar o trabalho, podendo usar a biblioteca.\", \"b\": \"Os alunos podem entregar o trabalho, devendo usar a biblioteca.\"}\n{\"id\": \"J109\", \"a\": \"No m\u00e1s de 3 personas por turno y al menos 2 turnos al d\u00eda.\", \"b\": \"Al menos 3 personas por turno y no m\u00e1s de 2 turnos al d\u00eda.\"}\n{\"id\": \"J110\", \"a\": \"Backups run nightly at 02:00 UTC.\", \"b\": \"Every night, at two in the morning UTC, the backup starts.\"}\n{\"id\": \"J111\", \"a\": \"Funcion\u00e1rios que n\u00e3o completaram o treinamento n\u00e3o podem operar a m\u00e1quina.\", \"b\": \"A m\u00e1quina n\u00e3o pode ser operada por funcion\u00e1rios que n\u00e3o completaram o treinamento.\"}\n{\"id\": \"J112\", \"a\": \"Solo si hay stock, confirma el pedido.\", \"b\": \"Si hay stock, confirma solo el pedido.\"}\n{\"id\": \"J113\", \"a\": \"El t\u00e9cnico debe revisar el equipo antes de instalarlo.\", \"b\": \"Antes de instalar el equipo, el t\u00e9cnico tiene la obligaci\u00f3n de revisarlo.\"}\n{\"id\": \"J114\", \"a\": \"Nunca dejes la puerta del laboratorio abierta, ni siquiera por un minuto.\", \"b\": \"La puerta del laboratorio debe permanecer cerrada en todo momento, incluso durante un minuto.\"}\n{\"id\": \"J115\", \"a\": \"May I bring a guest to the event? Yes, as long as the guest signs in at the desk.\", \"b\": \"May I bring a guest to the event? Yes, provided the guest checks in at the front desk.\"}\n{\"id\": \"J116\", \"a\": \"The supervisor signs off on every shift report.\", \"b\": \"Every shift report is signed off by the supervisor.\"}\n{\"id\": \"J117\", \"a\": \"Should I call the customer? Don't, email them.\", \"b\": \"Should I call the customer? Don't email them.\"}\n{\"id\": \"J118\", \"a\": \"Apart from the night shift, all employees get a 30-minute break.\", \"b\": \"All employees, other than those on the night shift, get a 30-minute break.\"}\n{\"id\": \"J119\", \"a\": \"Can I use the company card for taxis? Yes, up to $50 per ride.\", \"b\": \"Taxi rides may be paid with the company card, as long as each ride costs no more than $50.\"}\n{\"id\": \"J120\", \"a\": \"Puedes borrar todos los archivos temporales excepto los de la carpeta de respaldo.\", \"b\": \"Salvo los que est\u00e1n en la carpeta de respaldo, todos los archivos temporales se pueden borrar.\"}\n{\"id\": \"J121\", \"a\": \"Ning\u00fan visitante debe entrar a la sala de servidores sin escolta.\", \"b\": \"Los visitantes no pueden entrar a la sala de servidores si no van acompa\u00f1ados.\"}\n{\"id\": \"J122\", \"a\": \"Reinicie o servidor, mas n\u00e3o antes de avisar a equipe.\", \"b\": \"N\u00e3o reinicie o servidor, mas antes avise a equipe.\"}\n{\"id\": \"J123\", \"a\": \"Have the tenant pay the plumber and the landlord reimburse the tenant.\", \"b\": \"Have the landlord pay the plumber and the tenant reimburse the landlord.\"}\n{\"id\": \"J124\", \"a\": \"Los t\u00e9cnicos no deben apagar los servidores de prueba, excepto los del equipo de QA.\", \"b\": \"Los t\u00e9cnicos deben apagar los servidores de prueba, excepto los del equipo de QA.\"}\n{\"id\": \"J125\", \"a\": \"Posso trazer meu cachorro ao escrit\u00f3rio? N\u00e3o, exceto c\u00e3es-guia.\", \"b\": \"C\u00e3es n\u00e3o s\u00e3o permitidos no escrit\u00f3rio, a menos que sejam c\u00e3es-guia.\"}\n{\"id\": \"J126\", \"a\": \"Actualice todos los servidores excepto el de producci\u00f3n.\", \"b\": \"Actualice todos los servidores excepto el de respaldo.\"}\n{\"id\": \"J127\", \"a\": \"Ningu\u00e9m deve entrar na sala dos servidores sem crach\u00e1.\", \"b\": \"A entrada na sala dos servidores \u00e9 proibida a quem n\u00e3o tiver crach\u00e1.\"}\n{\"id\": \"J128\", \"a\": \"Withdrawals over 1,000 dollars need two signatures.\", \"b\": \"Withdrawals of 1,000 dollars or more need two signatures.\"}\n{\"id\": \"J129\", \"a\": \"Passwords must contain at least 12 characters.\", \"b\": \"A password shorter than 12 characters is not allowed.\"}\n{\"id\": \"J130\", \"a\": \"Submit the timesheet on Monday and the expense report by Friday.\", \"b\": \"Submit the timesheet by Monday and the expense report on Friday.\"}\n{\"id\": \"J131\", \"a\": \"Finance must be notified by the team lead.\", \"b\": \"The team lead must be notified by finance.\"}\n{\"id\": \"J132\", \"a\": \"Cierre el turno a las 18:00 y entregue las llaves a las 18:30.\", \"b\": \"A las 18:30 entregue las llaves; el turno se cierra a las 18:00.\"}\n{\"id\": \"J133\", \"a\": \"No reinicie el servicio y avise al equipo de soporte.\", \"b\": \"No reinicie el servicio ni avise al equipo de soporte.\"}\n{\"id\": \"J134\", \"a\": \"El alumno aprueba si obtiene al menos el 60 % en el examen final.\", \"b\": \"El alumno aprueba solo si obtiene m\u00e1s del 60 % en el examen final.\"}\n{\"id\": \"J135\", \"a\": \"Se cobrar\u00e1 el env\u00edo a todos los pedidos salvo a los superiores a 50 euros.\", \"b\": \"Todos los pedidos tienen gastos de env\u00edo, excepto aquellos que superan los 50 euros.\"}\n{\"id\": \"J136\", \"a\": \"Process no more than 500 records per batch.\", \"b\": \"Process no fewer than 500 records per batch.\"}\n{\"id\": \"J137\", \"a\": \"Pague o fornecedor at\u00e9 dia 10 e envie a nota at\u00e9 dia 15.\", \"b\": \"Pague o fornecedor at\u00e9 dia 15 e envie a nota at\u00e9 dia 10.\"}\n{\"id\": \"J138\", \"a\": \"Treinamento na ter\u00e7a \u00e0s 14h; avalia\u00e7\u00e3o na quinta \u00e0s 10h.\", \"b\": \"A avalia\u00e7\u00e3o ser\u00e1 na quinta, \u00e0s 10h, e o treinamento na ter\u00e7a, \u00e0s 14h.\"}\n{\"id\": \"J139\", \"a\": \"A equipe de suporte responde aos chamados abertos pelos clientes.\", \"b\": \"Os chamados que os clientes abrem s\u00e3o respondidos pela equipe de suporte.\"}\n{\"id\": \"J140\", \"a\": \"Solo se abre el expediente si hay una denuncia formal.\", \"b\": \"No se abre el expediente salvo que exista una denuncia formal.\"}\n{\"id\": \"J141\", \"a\": \"Can contractors access the VPN? Yes, but only from company devices.\", \"b\": \"Can contractors access the VPN? Yes, but only from company devices or from home.\"}\n{\"id\": \"J142\", \"a\": \"\u00bfHay que fichar al salir? S\u00ed, siempre antes de las 18:00.\", \"b\": \"\u00bfSe debe fichar al salir? Siempre, y antes de las 18:00.\"}\n{\"id\": \"J143\", \"a\": \"Before 9 a.m., check the alerts; after lunch, review the open tickets.\", \"b\": \"Review the open tickets after lunch, and check the alerts before 9 a.m.\"}\n{\"id\": \"J144\", \"a\": \"Avise o usu\u00e1rio e, se houver tr\u00eas tentativas falhas, bloqueie a conta.\", \"b\": \"Se houver tr\u00eas tentativas falhas, avise o usu\u00e1rio e bloqueie a conta.\"}\n{\"id\": \"J145\", \"a\": \"Se pueden usar los ascensores, excepto el de carga, en horas pico.\", \"b\": \"Se pueden usar los ascensores, excepto el de carga y el de servicio, en horas pico.\"}\n{\"id\": \"J146\", \"a\": \"Everyone except the contractor must complete the form.\", \"b\": \"Everyone except the contractors must complete the form.\"}\n{\"id\": \"J147\", \"a\": \"N\u00e3o use o elevador em caso de inc\u00eandio; utilize as escadas.\", \"b\": \"Em caso de inc\u00eandio, utilize as escadas e n\u00e3o o elevador.\"}\n{\"id\": \"J148\", \"a\": \"Staff must not photograph patients, but they may record the audio of consultations.\", \"b\": \"Staff must not photograph patients or record the audio of consultations.\"}\n{\"id\": \"J149\", \"a\": \"\u00bfPuedo cancelar el pedido despu\u00e9s de enviarlo? No, salvo que el cliente lo pida por escrito.\", \"b\": \"\u00bfPuedo cancelar el pedido despu\u00e9s de enviarlo? No. Solo si el cliente lo solicita por escrito.\"}\n{\"id\": \"J150\", \"a\": \"Can we ship on Friday? No, ship on Monday.\", \"b\": \"Can we ship on Friday? Not allowed. Ship on Monday instead.\"}\n{\"id\": \"J151\", \"a\": \"Mantenha pelo menos 2 metros de dist\u00e2ncia da m\u00e1quina.\", \"b\": \"N\u00e3o fique a menos de 2 metros da m\u00e1quina.\"}\n{\"id\": \"J152\", \"a\": \"Todos los empleados deben usar casco, salvo quienes trabajen en oficinas.\", \"b\": \"Quienes trabajan en oficinas no tienen que usar casco; los dem\u00e1s empleados s\u00ed deben usarlo.\"}\n{\"id\": \"J153\", \"a\": \"Back up the database at 2 a.m. and restart the server at 3 a.m.\", \"b\": \"Restart the server at 03:00, after backing up the database at 02:00.\"}\n{\"id\": \"J154\", \"a\": \"Todos deben fichar, excepto el personal de limpieza.\", \"b\": \"Salvo el personal de limpieza, todos deben fichar.\"}\n{\"id\": \"J155\", \"a\": \"Sell at least two units per customer.\", \"b\": \"Each customer must buy two units or more.\"}\n{\"id\": \"J156\", \"a\": \"\u00bfPuedo reiniciar el router? No, primero llame a soporte.\", \"b\": \"\u00bfPuedo reiniciar el router? No; llame primero a soporte.\"}\n{\"id\": \"J157\", \"a\": \"Drivers must not text, but they may use hands-free calls.\", \"b\": \"Drivers may not text, but they must use hands-free calls.\"}\n{\"id\": \"J158\", \"a\": \"La jefa aprueba los vi\u00e1ticos que presenta el empleado.\", \"b\": \"Los vi\u00e1ticos que presenta el empleado son aprobados por la jefa.\"}\n{\"id\": \"J159\", \"a\": \"Ningu\u00e9m, exceto o t\u00e9cnico, pode mexer no quadro el\u00e9trico.\", \"b\": \"Ningu\u00e9m pode mexer no quadro el\u00e9trico, exceto com o t\u00e9cnico.\"}\n{\"id\": \"J160\", \"a\": \"Can I return an opened item? Yes, provided it still has the seal.\", \"b\": \"Can I return an opened item? Yes, as long as the seal is intact.\"}\n{\"id\": \"J161\", \"a\": \"Todos los proveedores, menos los de Asia, deben entregar factura electr\u00f3nica.\", \"b\": \"Los proveedores de Asia quedan fuera; los dem\u00e1s deben entregar factura electr\u00f3nica.\"}\n{\"id\": \"J162\", \"a\": \"The manager approves the request from the employee.\", \"b\": \"The employee approves the request from the manager.\"}\n{\"id\": \"J163\", \"a\": \"Approve the budget before March 1 and publish it after March 15.\", \"b\": \"Approve the budget after March 1 and publish it before March 15.\"}\n{\"id\": \"J164\", \"a\": \"Notify the client within 24 hours and resolve the ticket within 72 hours.\", \"b\": \"Notify the client within 72 hours and resolve the ticket within 24 hours.\"}\n{\"id\": \"J165\", \"a\": \"If the temperature exceeds 30 \u00b0C, turn on the fan.\", \"b\": \"Turn on the fan whenever the temperature goes above 30 \u00b0C.\"}\n{\"id\": \"J166\", \"a\": \"\u00bfDebo avisar al jefe? No, avise a recursos humanos.\", \"b\": \"\u00bfDebo avisar al jefe? No avise a recursos humanos.\"}\n{\"id\": \"J167\", \"a\": \"The security team reviews every access request.\", \"b\": \"Every access request is reviewed by the security team.\"}\n{\"id\": \"J168\", \"a\": \"Los clientes pueden cancelar el pedido y deben conservar el recibo.\", \"b\": \"Los clientes deben cancelar el pedido y pueden conservar el recibo.\"}\n{\"id\": \"J169\", \"a\": \"No env\u00edes la factura hasta que el cliente confirme el pedido.\", \"b\": \"La factura no se env\u00eda antes de que el cliente confirme el pedido.\"}\n{\"id\": \"J170\", \"a\": \"Notifique al cliente el mismo d\u00eda que se apruebe el cambio.\", \"b\": \"El cliente debe recibir la notificaci\u00f3n el d\u00eda en que se apruebe el cambio.\"}\n{\"id\": \"J171\", \"a\": \"Unless the ticket is escalated, the agent handles it alone.\", \"b\": \"The agent handles the ticket alone, except when it is escalated.\"}\n{\"id\": \"J172\", \"a\": \"N\u00e3o envie o relat\u00f3rio ao cliente, mas arquive a c\u00f3pia interna.\", \"b\": \"Envie o relat\u00f3rio ao cliente, mas n\u00e3o arquive a c\u00f3pia interna.\"}\n{\"id\": \"J173\", \"a\": \"El cliente recibe la factura por correo electr\u00f3nico.\", \"b\": \"La factura le es enviada por correo electr\u00f3nico al cliente.\"}\n{\"id\": \"J174\", \"a\": \"No apague el equipo ni cierre sesi\u00f3n antes de terminar la copia.\", \"b\": \"Antes de terminar la copia, no apague el equipo y tampoco cierre sesi\u00f3n.\"}\n{\"id\": \"J175\", \"a\": \"Submit the report by Friday and pay the invoice on Monday.\", \"b\": \"Submit the report on Monday and pay the invoice by Friday.\"}\n{\"id\": \"J176\", \"a\": \"No apagues el equipo, pero desconecta el cable de red.\", \"b\": \"Apaga el equipo, pero no desconectes el cable de red.\"}\n{\"id\": \"J177\", \"a\": \"Os funcion\u00e1rios podem trabalhar de casa \u00e0s sextas, a n\u00e3o ser que haja reuni\u00e3o presencial.\", \"b\": \"\u00c0s sextas, salvo quando houver reuni\u00e3o presencial, \u00e9 permitido aos funcion\u00e1rios trabalhar de casa.\"}\n{\"id\": \"J178\", \"a\": \"Env\u00ede el informe a las 9:00 y cierre la caja a las 17:00.\", \"b\": \"Env\u00ede el informe a las 17:00 y cierre la caja a las 9:00.\"}\n{\"id\": \"J179\", \"a\": \"El supervisor revisa los informes que redactan los analistas.\", \"b\": \"Los analistas redactan los informes, y es el supervisor quien los revisa.\"}\n{\"id\": \"J180\", \"a\": \"\u00bfPuedo imprimir los contratos? S\u00ed, m\u00e1ximo 3 copias.\", \"b\": \"\u00bfPuedo imprimir los contratos? S\u00ed, m\u00ednimo 3 copias.\"}\n{\"id\": \"J181\", \"a\": \"Staff should log out at night and can leave the laptop at the office.\", \"b\": \"Staff can log out at night and should leave the laptop at the office.\"}\n{\"id\": \"J182\", \"a\": \"Fewer than 5 attempts are allowed per hour.\", \"b\": \"A maximum of 4 attempts per hour is allowed.\"}\n{\"id\": \"J183\", \"a\": \"Los empleados que no han firmado el contrato pueden acceder al sistema.\", \"b\": \"Los empleados que han firmado el contrato no pueden acceder al sistema.\"}\n{\"id\": \"J184\", \"a\": \"\u00bfHay que cifrar el disco? S\u00ed, con AES-256.\", \"b\": \"\u00bfHay que cifrar el disco? S\u00ed; use AES-256.\"}\n{\"id\": \"J185\", \"a\": \"Pack no more than 12 bottles per box and at least 6.\", \"b\": \"Each box must hold between 6 and 12 bottles, inclusive.\"}\n{\"id\": \"J186\", \"a\": \"Do I need approval? No, just notify your manager.\", \"b\": \"Do I need approval? Not required; simply let your manager know.\"}\n{\"id\": \"J187\", \"a\": \"S\u00f3 libere o acesso se o contrato estiver assinado.\", \"b\": \"Libere o acesso somente quando o contrato estiver assinado.\"}\n{\"id\": \"J188\", \"a\": \"Feche o caixa at\u00e9 as 18h, mas conte o dinheiro s\u00f3 \u00e0s 19h.\", \"b\": \"Feche o caixa \u00e0s 18h, mas conte o dinheiro at\u00e9 as 19h.\"}\n{\"id\": \"J189\", \"a\": \"The auditor must approve the change before the engineer deploys it.\", \"b\": \"The engineer must approve the change before the auditor deploys it.\"}\n{\"id\": \"J190\", \"a\": \"Debes usar casco en la obra; las gafas son opcionales.\", \"b\": \"En la obra el casco es obligatorio, mientras que puedes elegir si usas gafas.\"}\n{\"id\": \"J191\", \"a\": \"Voc\u00ea pode trabalhar remotamente, mas deve estar dispon\u00edvel das 10h \u00e0s 16h.\", \"b\": \"\u00c9 obrigat\u00f3rio estar dispon\u00edvel entre 10h e 16h; trabalhar remotamente \u00e9 permitido.\"}\n{\"id\": \"J192\", \"a\": \"Cuando la alarma suene, evacua el edificio por la escalera norte.\", \"b\": \"Si suena la alarma, evacua el edificio por la escalera norte.\"}\n{\"id\": \"J193\", \"a\": \"El cliente debe notificar al proveedor cualquier retraso.\", \"b\": \"El proveedor debe notificar al cliente cualquier retraso.\"}\n{\"id\": \"J194\", \"a\": \"Refunds are issued within 30 days unless the item was customized.\", \"b\": \"Refunds are issued within 30 days unless the item was damaged.\"}\n{\"id\": \"J195\", \"a\": \"Managers may approve leave, provided HR is informed.\", \"b\": \"Leave can be approved by a manager as long as HR is told.\"}\n{\"id\": \"J196\", \"a\": \"Do not delete logs that were archived last week.\", \"b\": \"Delete logs that were not archived last week.\"}\n{\"id\": \"J197\", \"a\": \"Cierre la caja a las 20:00 y limpie el local despu\u00e9s de las 21:00.\", \"b\": \"Cierre la caja despu\u00e9s de las 20:00 y limpie el local a las 21:00.\"}\n{\"id\": \"J198\", \"a\": \"Cada supervisor debe revisar al menos 10 pedidos por d\u00eda.\", \"b\": \"Cada supervisor debe revisar a lo sumo 10 pedidos por d\u00eda.\"}\n{\"id\": \"J199\", \"a\": \"Todos os relat\u00f3rios devem ser revisados, exceto os mensais.\", \"b\": \"Com exce\u00e7\u00e3o dos relat\u00f3rios mensais, todos devem passar por revis\u00e3o.\"}\n{\"id\": \"J200\", \"a\": \"El 15 de cada mes se paga la n\u00f3mina; el d\u00eda 30, los proveedores.\", \"b\": \"A los proveedores se les paga el 30 de cada mes y la n\u00f3mina se paga el d\u00eda 15.\"}\n{\"id\": \"J201\", \"a\": \"Solo el administrador puede modificar los permisos.\", \"b\": \"Los permisos \u00fanicamente pueden ser modificados por el administrador.\"}\n{\"id\": \"J202\", \"a\": \"Staff must lock the cabinet, but they may leave the key with a supervisor.\", \"b\": \"Staff may lock the cabinet, but they must leave the key with a supervisor.\"}\n{\"id\": \"J203\", \"a\": \"Users can reset their password but must not reuse the old one.\", \"b\": \"Users are allowed to reset their password; reusing the old one is forbidden.\"}\n{\"id\": \"J204\", \"a\": \"Aceite no m\u00ednimo cinco candidatos por vaga.\", \"b\": \"Aceite no m\u00e1ximo cinco candidatos por vaga.\"}\n{\"id\": \"J205\", \"a\": \"O cliente deve assinar o termo e pode levar uma c\u00f3pia.\", \"b\": \"O cliente pode assinar o termo e deve levar uma c\u00f3pia.\"}\n{\"id\": \"J206\", \"a\": \"El supervisor firma el formulario que el t\u00e9cnico entrega.\", \"b\": \"El t\u00e9cnico firma el formulario que el supervisor entrega.\"}\n{\"id\": \"J207\", \"a\": \"Puede retirar el reembolso en efectivo, aunque se recomienda la transferencia.\", \"b\": \"Es posible cobrar el reembolso en efectivo; no obstante, la transferencia es lo recomendado.\"}\n{\"id\": \"J208\", \"a\": \"No uses guantes de l\u00e1tex y l\u00e1vate las manos.\", \"b\": \"No uses guantes de l\u00e1tex ni te laves las manos.\"}\n{\"id\": \"J209\", \"a\": \"Posso usar o cart\u00e3o corporativo para hotel? Sim, at\u00e9 300 reais por noite.\", \"b\": \"Posso usar o cart\u00e3o corporativo para hotel? Sim, a partir de 300 reais por noite.\"}\n{\"id\": \"J210\", \"a\": \"Solo si el paciente firma el consentimiento se realiza el procedimiento.\", \"b\": \"El procedimiento se realiza \u00fanicamente cuando el paciente firma el consentimiento.\"}\n{\"id\": \"J211\", \"a\": \"Visitors must sign in at reception; they may use the guest Wi-Fi.\", \"b\": \"Visitors are allowed to use the guest Wi-Fi, but they are required to sign in at reception.\"}\n{\"id\": \"J212\", \"a\": \"Do not open attachments from unknown senders, but report them to security.\", \"b\": \"Report attachments from unknown senders to security, and do not open them.\"}\n{\"id\": \"J213\", \"a\": \"No se permite fumar en la sala de servidores.\", \"b\": \"Est\u00e1 prohibido fumar en la sala de servidores.\"}\n{\"id\": \"J214\", \"a\": \"Reserve more than two parking spaces only with approval.\", \"b\": \"Reserve fewer than two parking spaces only with approval.\"}\n{\"id\": \"J215\", \"a\": \"A cozinha separa os pedidos que o gar\u00e7om anota.\", \"b\": \"O gar\u00e7om separa os pedidos que a cozinha anota.\"}\n{\"id\": \"J216\", \"a\": \"Restart the service only if the health check fails.\", \"b\": \"Restart the service if the health check fails.\"}\n{\"id\": \"J217\", \"a\": \"Employees must report incidents and may request a copy of the report.\", \"b\": \"Employees may report incidents and must request a copy of the report.\"}\n{\"id\": \"J218\", \"a\": \"Do not restart the servers that host the database during backups.\", \"b\": \"Restart the servers that do not host the database during backups.\"}\n{\"id\": \"J219\", \"a\": \"Never share your password with anyone, including IT staff.\", \"b\": \"Your password must not be shared with anybody \u2014 IT staff included.\"}\n{\"id\": \"J220\", \"a\": \"Voc\u00ea pode cancelar a reserva sem multa, mas deve avisar com 48 horas.\", \"b\": \"Voc\u00ea deve cancelar a reserva sem multa, mas pode avisar com 48 horas.\"}\n{\"id\": \"J221\", \"a\": \"The auditor reviews the ledger, and the CFO signs it.\", \"b\": \"The ledger is reviewed by the auditor and signed by the CFO.\"}\n{\"id\": \"J222\", \"a\": \"Preencha o formul\u00e1rio at\u00e9 quinta e envie o comprovante at\u00e9 sexta.\", \"b\": \"At\u00e9 sexta, envie o comprovante; o formul\u00e1rio deve ser preenchido at\u00e9 quinta.\"}\n{\"id\": \"J223\", \"a\": \"Se permiten como m\u00e1ximo cinco intentos de inicio de sesi\u00f3n.\", \"b\": \"No se admiten m\u00e1s de cinco intentos de inicio de sesi\u00f3n.\"}\n{\"id\": \"J224\", \"a\": \"Todos devem usar crach\u00e1, a n\u00e3o ser os visitantes escoltados.\", \"b\": \"Os visitantes escoltados s\u00e3o os \u00fanicos que podem dispensar o crach\u00e1.\"}\n{\"id\": \"J225\", \"a\": \"A menos que llueva, el evento ser\u00e1 al aire libre.\", \"b\": \"El evento se har\u00e1 al aire libre salvo que llueva.\"}\n{\"id\": \"J226\", \"a\": \"O cliente paga a taxa e o banco emite o comprovante.\", \"b\": \"O comprovante \u00e9 emitido pelo banco, depois que o cliente paga a taxa.\"}\n{\"id\": \"J227\", \"a\": \"Do I sign the form? No; your manager signs it.\", \"b\": \"You don't sign the form \u2014 your manager does.\"}\n{\"id\": \"J228\", \"a\": \"Approve invoices below 500 dollars; do not notify finance.\", \"b\": \"Do not approve invoices below 500 dollars; notify finance.\"}\n{\"id\": \"J229\", \"a\": \"No se permite fumar en el dep\u00f3sito ni comer en la oficina de carga.\", \"b\": \"En el dep\u00f3sito est\u00e1 prohibido fumar, y en la oficina de carga est\u00e1 prohibido comer.\"}\n{\"id\": \"J230\", \"a\": \"Desative todas as contas inativas, exceto as de administradores.\", \"b\": \"Desative todas as contas, exceto as inativas de administradores.\"}\n{\"id\": \"J231\", \"a\": \"You should back up the laptop before travel, and you can ask IT for a loaner.\", \"b\": \"Before travelling, you are advised to back up your laptop, and IT may lend you one.\"}\n{\"id\": \"J232\", \"a\": \"Puedes cancelar la reserva y debes avisar al hotel.\", \"b\": \"Debes cancelar la reserva y puedes avisar al hotel.\"}\n{\"id\": \"J233\", \"a\": \"O contador deve ser informado pelo auditor sobre qualquer erro.\", \"b\": \"O contador deve informar o auditor sobre qualquer erro.\"}\n{\"id\": \"J234\", \"a\": \"Os dados ser\u00e3o apagados se o cliente pedir, mas n\u00e3o antes de 30 dias.\", \"b\": \"Os dados ser\u00e3o apagados se o cliente pedir, mesmo antes de 30 dias.\"}\n{\"id\": \"J235\", \"a\": \"No m\u00e1ximo 3 dispositivos podem estar conectados ao mesmo tempo.\", \"b\": \"At\u00e9 tr\u00eas dispositivos podem estar conectados simultaneamente.\"}\n{\"id\": \"J236\", \"a\": \"Cierra todas las tiendas el domingo, salvo la del centro.\", \"b\": \"Cierra todas las tiendas el domingo, salvo las del centro.\"}\n{\"id\": \"J237\", \"a\": \"Rotate the keys within 24 hours of an incident, but report it immediately.\", \"b\": \"Rotate the keys within 24 hours of an incident, but report it within 24 hours.\"}\n{\"id\": \"J238\", \"a\": \"May I leave early on Friday? Yes, but not before 3 p.m.\", \"b\": \"May I leave early on Friday? Yes, but not after 3 p.m.\"}\n{\"id\": \"J239\", \"a\": \"O reembolso s\u00f3 \u00e9 aceito para compras acima de 50 reais.\", \"b\": \"Compras de 50 reais ou menos n\u00e3o s\u00e3o eleg\u00edveis a reembolso.\"}\n{\"id\": \"J240\", \"a\": \"Unless the customer objects, send the invoice by email.\", \"b\": \"Send the invoice by email, provided the customer does not object.\"}\n{\"id\": \"J241\", \"a\": \"Everyone except the night staff must complete the safety course by Friday.\", \"b\": \"Apart from the night crew, all employees have to finish the safety course by Friday.\"}\n{\"id\": \"J242\", \"a\": \"El personal debe usar casco y puede traer su propio chaleco.\", \"b\": \"Es obligatorio que el personal use casco; traer chaleco propio es opcional.\"}\n{\"id\": \"J243\", \"a\": \"Si el cliente lo pide, env\u00eda la factura y cierra el ticket.\", \"b\": \"Env\u00eda la factura si el cliente lo pide, y cierra el ticket.\"}\n{\"id\": \"J244\", \"a\": \"Keep at least three copies of each backup, on two media types.\", \"b\": \"Keep at most three copies of each backup, on two media types.\"}\n{\"id\": \"J245\", \"a\": \"Recursos Humanos entrevista a los candidatos que selecciona el gerente.\", \"b\": \"Los candidatos seleccionados por el gerente son entrevistados por Recursos Humanos.\"}\n{\"id\": \"J246\", \"a\": \"Entregue el informe antes del viernes a las 17:00.\", \"b\": \"El informe debe estar entregado a m\u00e1s tardar el viernes a las cinco de la tarde.\"}\n{\"id\": \"J247\", \"a\": \"\u00bfPuedo llevar mi perro a la oficina? S\u00ed, pero solo uno por equipo.\", \"b\": \"\u00bfPuedo llevar mi perro a la oficina? S\u00ed, pero solo uno por persona.\"}\n{\"id\": \"J248\", \"a\": \"Fa\u00e7a backup de todos os diret\u00f3rios, exceto /tmp.\", \"b\": \"Fa\u00e7a backup de todas as pastas, menos a /tmp.\"}\n{\"id\": \"J249\", \"a\": \"Se a meta for atingida, o b\u00f4nus \u00e9 pago a todos os funcion\u00e1rios.\", \"b\": \"Se a meta for atingida, o b\u00f4nus \u00e9 pago aos funcion\u00e1rios ativos.\"}\n{\"id\": \"J250\", \"a\": \"O estagi\u00e1rio n\u00e3o deve assinar contratos sem a presen\u00e7a de um s\u00f3cio.\", \"b\": \"O estagi\u00e1rio deve assinar contratos sem a presen\u00e7a de um s\u00f3cio.\"}\n{\"id\": \"J251\", \"a\": \"A reuni\u00e3o come\u00e7a \u00e0s 9h e termina antes das 12h.\", \"b\": \"Ela come\u00e7a \u00e0s nove da manh\u00e3 e termina antes do meio-dia.\"}\n{\"id\": \"J252\", \"a\": \"Cada turno precisa de pelo menos 4 enfermeiros e no m\u00e1ximo 2 estagi\u00e1rios.\", \"b\": \"Em cada turno, s\u00e3o necess\u00e1rios 4 enfermeiros ou mais e at\u00e9 2 estagi\u00e1rios.\"}\n{\"id\": \"J253\", \"a\": \"Do not submit the invoice until the purchase order is approved, and do not pay it before the goods arrive.\", \"b\": \"Do not submit the invoice until the purchase order is approved, and pay it before the goods arrive.\"}\n{\"id\": \"J254\", \"a\": \"All staff must attend the drill, apart from employees on medical leave.\", \"b\": \"Except for employees on medical leave, every staff member must attend the drill.\"}\n{\"id\": \"J255\", \"a\": \"Los nuevos empleados pueden usar la impresora, pero deben pedir el c\u00f3digo.\", \"b\": \"Los nuevos empleados deben usar la impresora, pero pueden pedir el c\u00f3digo.\"}\n{\"id\": \"J256\", \"a\": \"Customers should be told by the agent, not the bot, about refunds.\", \"b\": \"Customers should be told by the bot, not the agent, about refunds.\"}\n{\"id\": \"J257\", \"a\": \"O auditor revisa os registros que o administrador gerou.\", \"b\": \"O administrador revisa os registros que o auditor gerou.\"}\n{\"id\": \"J258\", \"a\": \"\u00bfSe puede usar el coche de la empresa para viajes personales? No, salvo con autorizaci\u00f3n escrita.\", \"b\": \"\u00bfSe puede usar el coche de la empresa para viajes personales? No, a menos que haya autorizaci\u00f3n escrita.\"}\n{\"id\": \"J259\", \"a\": \"Se requieren m\u00e1s de 2 firmas para montos superiores a 500 euros.\", \"b\": \"Los montos de m\u00e1s de 500 euros requieren 3 firmas o m\u00e1s.\"}\n{\"id\": \"J260\", \"a\": \"On Mondays, water the plants; on Wednesdays, take out the recycling.\", \"b\": \"Take out the recycling every Wednesday and water the plants every Monday.\"}\n{\"id\": \"J261\", \"a\": \"Os visitantes podem tirar fotos, mas devem pedir autoriza\u00e7\u00e3o para filmar.\", \"b\": \"Tirar fotos \u00e9 permitido aos visitantes; para filmar, \u00e9 obrigat\u00f3rio pedir autoriza\u00e7\u00e3o.\"}\n{\"id\": \"J262\", \"a\": \"Usted debe firmar el acta y puede enviar una copia por correo.\", \"b\": \"Usted puede enviar una copia del acta por correo, pero debe firmarla.\"}\n{\"id\": \"J263\", \"a\": \"Un m\u00e1ximo de dos invitados por reserva.\", \"b\": \"Cada reserva admite hasta dos invitados.\"}\n{\"id\": \"J264\", \"a\": \"O diretor aprova os pedidos que os gerentes enviam.\", \"b\": \"Os pedidos enviados pelos gerentes s\u00e3o aprovados pelo diretor.\"}\n{\"id\": \"J265\", \"a\": \"Operators restart the service after the patch is applied.\", \"b\": \"After the patch is applied, the service is restarted by operators.\"}\n{\"id\": \"J266\", \"a\": \"N\u00e3o deixe a porta aberta quando sair do laborat\u00f3rio.\", \"b\": \"Deixe a porta aberta quando n\u00e3o sair do laborat\u00f3rio.\"}\n{\"id\": \"J267\", \"a\": \"Mais de 20 colaboradores devem fazer o treinamento.\", \"b\": \"Pelo menos 20 colaboradores devem fazer o treinamento.\"}\n{\"id\": \"J268\", \"a\": \"Applicants may submit a cover letter and must attach a resume.\", \"b\": \"A resume must be attached by applicants, who may also submit a cover letter.\"}\n{\"id\": \"J269\", \"a\": \"Posso sair mais cedo? Sim, desde que avise o gerente.\", \"b\": \"Posso sair mais cedo? Sim \u2014 contanto que avise o gerente.\"}\n{\"id\": \"J270\", \"a\": \"Call the on-call engineer when the alert fires, unless it's a weekend.\", \"b\": \"Call the on-call engineer when the alert fires, even if it's a weekend.\"}\n{\"id\": \"K001\", \"a\": \"Cobra el pedido sin contar los env\u00edos gratuitos.\", \"b\": \"Cobra los pedidos sin contar el env\u00edo gratuito.\"}\n{\"id\": \"K002\", \"a\": \"Say \\\"No\\\" and then \\\"more than ten\\\".\", \"b\": \"Say \\\"No more than ten\\\".\"}\n{\"id\": \"K003\", \"a\": \"Abra a v\u00e1lvula.\", \"b\": \"Feche a v\u00e1lvula.\"}\n{\"id\": \"K004\", \"a\": \"Staff should not share the password, may share the username.\", \"b\": \"Staff should share the password, may not share the username.\"}\n{\"id\": \"K005\", \"a\": \"Keep at most 5 backups.\", \"b\": \"Guarde no m\u00e1ximo 5 c\u00f3pias.\"}\n{\"id\": \"K006\", \"a\": \"Accept ages 18 and over.\", \"b\": \"Accept ages over 18.\"}\n{\"id\": \"K007\", \"a\": \"Keep fewer than 5 backups.\", \"b\": \"Keep less than 5 backups.\"}\n{\"id\": \"K008\", \"a\": \"The report that is not signed must be returned.\", \"b\": \"The report that is signed must not be returned.\"}\n{\"id\": \"K009\", \"a\": \"Nunca, bajo ninguna circunstancia, borres el log.\", \"b\": \"Nunca bajo ninguna circunstancia borres el log.\"}\n{\"id\": \"K010\", \"a\": \"Permite el acceso si el usuario inici\u00f3 sesi\u00f3n.\", \"b\": \"Permite el acceso solo si el usuario inici\u00f3 sesi\u00f3n.\"}\n{\"id\": \"K011\", \"a\": \"Never restart the node; drain the queue first.\", \"b\": \"Restart the node; never drain the queue first.\"}\n{\"id\": \"K012\", \"a\": \"Use a chave de teste, n\u00e3o a de produ\u00e7\u00e3o.\", \"b\": \"Use a chave de teste e n\u00e3o a de produ\u00e7\u00e3o.\"}\n{\"id\": \"K013\", \"a\": \"Reinicie todos os servidores, excetuando o de teste.\", \"b\": \"Reinicie todos os servidores, excetuando os de teste.\"}\n{\"id\": \"K014\", \"a\": \"Disable every account barring the service account.\", \"b\": \"Disable every account barring the service accounts.\"}\n{\"id\": \"K015\", \"a\": \"Guarde pelo menos 5 c\u00f3pias.\", \"b\": \"Guarde 5 ou mais c\u00f3pias.\"}\n{\"id\": \"K016\", \"a\": \"Save, but delete, but never archive.\", \"b\": \"Save, but never delete, but archive.\"}\n{\"id\": \"K017\", \"a\": \"Do not save the file, but send the copy.\", \"b\": \"Do not save the file; send the copy.\"}\n{\"id\": \"K018\", \"a\": \"Abierto de lunes a viernes.\", \"b\": \"Abierto de lunes a s\u00e1bado.\"}\n{\"id\": \"K019\", \"a\": \"The limit is 100 requests per minute.\", \"b\": \"The limit is 100 requests per hour.\"}\n{\"id\": \"K020\", \"a\": \"Don't delete anything but the logs.\", \"b\": \"Delete anything but don't touch the logs.\"}\n{\"id\": \"K021\", \"a\": \"Cierra todos los tickets excepto el urgente.\", \"b\": \"Cierra todos los tickets excepto los urgentes.\"}\n{\"id\": \"K022\", \"a\": \"Never delete the file, although it is old.\", \"b\": \"Although it is old, never delete the file.\"}\n{\"id\": \"K023\", \"a\": \"Restart all servers except the test servers.\", \"b\": \"Restart all servers, except the test servers!\"}\n{\"id\": \"K024\", \"a\": \"Confirm that the backup did not fail.\", \"b\": \"Confirm the backup did not fail.\"}\n{\"id\": \"K025\", \"a\": \"Step 2: Do not format the volume, mount the share.\", \"b\": \"Step 2: Format the volume, do not mount the share.\"}\n{\"id\": \"K026\", \"a\": \"Enable the firewall.\", \"b\": \"Turn on the firewall.\"}\n{\"id\": \"K027\", \"a\": \"Revise o c\u00f3digo, n\u00e3o implante o servi\u00e7o.\", \"b\": \"N\u00e3o revise o c\u00f3digo, implante o servi\u00e7o.\"}\n{\"id\": \"K028\", \"a\": \"Audit all departments except the sales team.\", \"b\": \"Audit all departments except the sales teams.\"}\n{\"id\": \"K029\", \"a\": \"Keep logs? No,fewer than five days.\", \"b\": \"Keep logs? No fewer than five days.\"}\n{\"id\": \"K030\", \"a\": \"Reinicie todas as maquinas, com excecao da de teste.\", \"b\": \"Reinicie todas as maquinas, com excecao das de teste.\"}\n{\"id\": \"K031\", \"a\": \"Reintenta si falla.\", \"b\": \"Reintenta si funciona.\"}\n{\"id\": \"K032\", \"a\": \"Reinicia todo excepto los servidores de prueba.\", \"b\": \"Reinicia todo con excepci\u00f3n de los servidores de prueba.\"}\n{\"id\": \"K033\", \"a\": \"Feche os chamados que n\u00e3o t\u00eam respons\u00e1vel.\", \"b\": \"Feche os chamados sem respons\u00e1vel.\"}\n{\"id\": \"K034\", \"a\": \"Do not pay the invoice, send the receipt.\", \"b\": \"Do not pay the invoice. Send the receipt.\"}\n{\"id\": \"K035\", \"a\": \"Do not approve the invoice, reject the refund, and escalate.\", \"b\": \"Approve the invoice, do not reject the refund, and escalate.\"}\n{\"id\": \"K036\", \"a\": \"Debes hacer la copia.\", \"b\": \"Tienes que hacer la copia.\"}\n{\"id\": \"K037\", \"a\": \"Nao envie o relatorio enquanto revisa o contrato.\", \"b\": \"Envie o relatorio enquanto nao revisa o contrato.\"}\n{\"id\": \"K038\", \"a\": \"Mantemos os logs? Em hipotese alguma, menos de cinco dias.\", \"b\": \"Mantemos os logs? Em hipotese alguma menos de cinco dias.\"}\n{\"id\": \"K039\", \"a\": \"Ship every order minus the gift order.\", \"b\": \"Ship every order minus the gift orders.\"}\n{\"id\": \"K040\", \"a\": \"Deploy to all regions excluding the east region.\", \"b\": \"Deploy to all regions except the east region.\"}\n{\"id\": \"K041\", \"a\": \"Nao exporte a base, pois arquive o relatorio.\", \"b\": \"Exporte a base, pois nao arquive o relatorio.\"}\n{\"id\": \"K042\", \"a\": \"Restart everything except the others.\", \"b\": \"Restart everything except the other.\"}\n{\"id\": \"K043\", \"a\": \"Envie a todos exceto a gerente.\", \"b\": \"Envie a todos exceto as gerentes.\"}\n{\"id\": \"K044\", \"a\": \"Envie a todos exceto a gerente.\", \"b\": \"Envie a todos, exceto a gerente.\"}\n{\"id\": \"K045\", \"a\": \"It is recommended to restart.\", \"b\": \"It is required to restart.\"}\n{\"id\": \"K046\", \"a\": \"No, menos de tres.\", \"b\": \"No menos de tres.\"}\n{\"id\": \"K047\", \"a\": \"Apague o arquivo exceto o cabe\u00e7alho.\", \"b\": \"Apague o arquivo exceto os cabe\u00e7alhos.\"}\n{\"id\": \"K048\", \"a\": \"Delete all files except the report.\", \"b\": \"Delete all files other than the report.\"}\n{\"id\": \"K049\", \"a\": \"Employees who never arrive late get a bonus.\", \"b\": \"Employees who arrive late never get a bonus.\"}\n{\"id\": \"K050\", \"a\": \"Apague todos os arquivos exceto o relat\u00f3rio.\", \"b\": \"Apague todos os arquivos exceto os relat\u00f3rios.\"}\n{\"id\": \"K051\", \"a\": \"Reply no, fewer than five days.\", \"b\": \"Reply no, fewer than five days!\"}\n{\"id\": \"K052\", \"a\": \"Do not delete the file.\", \"b\": \"N\u00e3o apague o arquivo.\"}\n{\"id\": \"K053\", \"a\": \"Ensure the script, which does not delete files, runs first.\", \"b\": \"Ensure the script, which deletes files, does not run first.\"}\n{\"id\": \"K054\", \"a\": \"Costs under 100 USD? No, 100 USD or more.\", \"b\": \"Costs under 100 USD? No 100 USD or more.\"}\n{\"id\": \"K055\", \"a\": \"Should I save? No, it is locked.\", \"b\": \"Should I save? No; it is locked.\"}\n{\"id\": \"K056\", \"a\": \"O sistema n\u00e3o deve permitir que os usu\u00e1rios que s\u00e3o administradores apaguem.\", \"b\": \"O sistema deve permitir que os usu\u00e1rios que n\u00e3o s\u00e3o administradores apaguem.\"}\n{\"id\": \"K057\", \"a\": \"Reply: **No**\\nAt most three.\", \"b\": \"Reply: **Yes**\\nAt most three.\"}\n{\"id\": \"K058\", \"a\": \"Delete the backups excluding the weekly backup.\", \"b\": \"Delete the backups excluding the weekly backups.\"}\n{\"id\": \"K059\", \"a\": \"Confirme que o backup n\u00e3o falhou.\", \"b\": \"Confirme que n\u00e3o falhou o backup.\"}\n{\"id\": \"K060\", \"a\": \"Entregue antes de sexta.\", \"b\": \"Entregue depois de sexta.\"}\n{\"id\": \"K061\", \"a\": \"Keep logs for no fewer than five days.\", \"b\": \"Keep logs for at least five days.\"}\n{\"id\": \"K062\", \"a\": \"N\u00e3o rejeite nenhuma das solicita\u00e7\u00f5es.\", \"b\": \"N\u00e3o rejeite todas as solicita\u00e7\u00f5es.\"}\n{\"id\": \"K063\", \"a\": \"No, por favor, desactives la alarma.\", \"b\": \"No por favor desactives la alarma.\"}\n{\"id\": \"K064\", \"a\": \"Never, under any circumstances, delete the log.\", \"b\": \"Never under any circumstances delete the log.\"}\n{\"id\": \"K065\", \"a\": \"Approve all requests except the largest one.\", \"b\": \"Approve all requests except the largest ones.\"}\n{\"id\": \"K066\", \"a\": \"Los empleados que nunca llegan tarde reciben un bono.\", \"b\": \"Los empleados que llegan tarde nunca reciben un bono.\"}\n{\"id\": \"K067\", \"a\": \"N\u00e3o envie o relat\u00f3rio ao cliente, mas envie a fatura.\", \"b\": \"N\u00e3o envie o relat\u00f3rio ao cliente; envie a fatura.\"}\n{\"id\": \"K068\", \"a\": \"Submit before Friday.\", \"b\": \"Submit on Friday.\"}\n{\"id\": \"K069\", \"a\": \"Salve o arquivo, mas nunca apague a c\u00f3pia.\", \"b\": \"Salve o arquivo, contudo nunca apague a c\u00f3pia.\"}\n{\"id\": \"K070\", \"a\": \"Confirm that the backup did not fail.\", \"b\": \"Do not confirm that the backup failed.\"}\n{\"id\": \"K071\", \"a\": \"Nunca salve, embora apague a c\u00f3pia.\", \"b\": \"Salve, embora nunca apague a c\u00f3pia.\"}\n{\"id\": \"K072\", \"a\": \"La fecha l\u00edmite es el 4 de marzo.\", \"b\": \"La fecha l\u00edmite es el 3 de abril.\"}\n{\"id\": \"K073\", \"a\": \"Agrega al usuario al grupo.\", \"b\": \"Quita al usuario del grupo.\"}\n{\"id\": \"K074\", \"a\": \"Reinicie todos os servidores, ignorando o de teste.\", \"b\": \"Reinicie todos os servidores, ignorando os de teste.\"}\n{\"id\": \"K075\", \"a\": \"Never mute the alert and throttle the API; escalate to oncall.\", \"b\": \"Mute the alert and never throttle the API; escalate to oncall.\"}\n{\"id\": \"K076\", \"a\": \"Restart the servers except test-server.\", \"b\": \"Restart the servers except test-servers.\"}\n{\"id\": \"K077\", \"a\": \"Processe primeiro os arquivos grandes.\", \"b\": \"Processe primeiro os arquivos pequenos.\"}\n{\"id\": \"K078\", \"a\": \"Keep logs? No\u00a0\u00a0\u2022\u00a0fewer than five days.\", \"b\": \"Keep logs? No fewer than five days.\"}\n{\"id\": \"K079\", \"a\": \"You must back up.\", \"b\": \"Voc\u00ea pode fazer o backup.\"}\n{\"id\": \"K080\", \"a\": \"Delete the old reports and keep the rest.\", \"b\": \"Keep the rest and delete the old reports.\"}\n{\"id\": \"K081\", \"a\": \"No, at most three.\", \"b\": \"No, three at most.\"}\n{\"id\": \"K082\", \"a\": \"Pague em at\u00e9 30 dias.\", \"b\": \"Pague depois de 30 dias.\"}\n{\"id\": \"K083\", \"a\": \"Never email the client, notify the manager.\", \"b\": \"Email the client, never notify the manager.\"}\n{\"id\": \"K084\", \"a\": \"Ensure that we never expose the key and rotate the token.\", \"b\": \"Ensure that we expose the key and never rotate the token.\"}\n{\"id\": \"K085\", \"a\": \"No emita la factura y registre el pago.\", \"b\": \"Emita la factura y no registre el pago.\"}\n{\"id\": \"K086\", \"a\": \"Envie tudo menos o contrato.\", \"b\": \"Envie tudo exceto o contrato.\"}\n{\"id\": \"K087\", \"a\": \"You must not submit the form.\", \"b\": \"You may submit the form.\"}\n{\"id\": \"K088\", \"a\": \"Keep at least 5 backups.\", \"b\": \"Keep 5 or more backups.\"}\n{\"id\": \"K089\", \"a\": \"Not only Ana but Luis must sign.\", \"b\": \"Only Ana must sign.\"}\n{\"id\": \"K090\", \"a\": \"Delete every account except mine.\", \"b\": \"Delete every account except ours.\"}\n{\"id\": \"K091\", \"a\": \"N\u00e3o, menos de cinco dias.\", \"b\": \"N\u00e3o: menos de cinco dias.\"}\n{\"id\": \"K092\", \"a\": \"Invite all staff except the child.\", \"b\": \"Invite all staff except the children.\"}\n{\"id\": \"K093\", \"a\": \"Restart two servers except the test.\", \"b\": \"Restart two server except the tests.\"}\n{\"id\": \"K094\", \"a\": \"Reinicia todos los nodos salvo el maestro.\", \"b\": \"Reinicia todos los nodos salvo los maestros.\"}\n{\"id\": \"K095\", \"a\": \"I know that Ana did not approve it.\", \"b\": \"I do not know that Ana approved it.\"}\n{\"id\": \"K096\", \"a\": \"Never save the file, delete the copy, but archive it.\", \"b\": \"Save the file, never delete the copy, but archive it.\"}\n{\"id\": \"K097\", \"a\": \"Nobody must approve the plan and the budget.\", \"b\": \"Nobody must approve the plan but everybody must approve the budget.\"}\n{\"id\": \"K098\", \"a\": \"Resposta: **N\u00e3o**\\nMenos de cinco dias.\", \"b\": \"Resposta: N\u00e3o menos de cinco dias.\"}\n{\"id\": \"K099\", \"a\": \"Allow access if the user is logged in.\", \"b\": \"Allow access only if the user is logged in.\"}\n{\"id\": \"K100\", \"a\": \"The report covers every region except the Americas and Asia.\", \"b\": \"The report covers every region except Asia and the Americas.\"}\n{\"id\": \"K101\", \"a\": \"I never ping the client and I quarantine the host.\", \"b\": \"I ping the client and I never quarantine the host.\"}\n{\"id\": \"K102\", \"a\": \"Guarda como m\u00e1ximo 5 copias.\", \"b\": \"Guarda no m\u00e1s de 5 copias.\"}\n{\"id\": \"K103\", \"a\": \"Delete everything but the test, never save.\", \"b\": \"Delete everything, never but the test save.\"}\n{\"id\": \"K104\", \"a\": \"No, fewer than five days.\", \"b\": \"No. Fewer than five days.\"}\n{\"id\": \"K105\", \"a\": \"Should I drop the table? Do not, drop the view.\", \"b\": \"Should I drop the table? Do not drop the view.\"}\n{\"id\": \"K106\", \"a\": \"Only the manager signs the invoice.\", \"b\": \"The manager only signs the invoice.\"}\n{\"id\": \"K107\", \"a\": \"Nunca, en ningun caso, desactives la alarma.\", \"b\": \"Nunca en ningun caso desactives la alarma.\"}\n{\"id\": \"K108\", \"a\": \"Do not ever restart; always drain first.\", \"b\": \"Always restart; do not ever drain first.\"}\n{\"id\": \"K109\", \"a\": \"You must back up the data.\", \"b\": \"You have to back up the data.\"}\n{\"id\": \"K110\", \"a\": \"Keep logs? Nay, fewer than five days.\", \"b\": \"Keep logs? Nay fewer than five days.\"}\n{\"id\": \"K111\", \"a\": \"Keep logs? No > fewer than five days.\", \"b\": \"Keep logs? No fewer than five days.\"}\n{\"id\": \"K112\", \"a\": \"Restart all servers but that.\", \"b\": \"Restart all servers but those.\"}\n{\"id\": \"K113\", \"a\": \"Apague o backup mais antigo.\", \"b\": \"Apague o backup mais recente.\"}\n{\"id\": \"K114\", \"a\": \"S\u00ed, al menos dos.\", \"b\": \"S\u00ed. Al menos dos.\"}\n{\"id\": \"K115\", \"a\": \"Delete every log except the last two.\", \"b\": \"Delete every log except the last three.\"}\n{\"id\": \"K116\", \"a\": \"Keep logs? Nay, fewer than five days.\", \"b\": \"Keep logs? Nay fewer than five days.\"}\n{\"id\": \"K117\", \"a\": \"N\u00e3o, apenas o gerente.\", \"b\": \"N\u00e3o apenas o gerente.\"}\n{\"id\": \"K118\", \"a\": \"Never share the password; rotate the key.\", \"b\": \"Share the password; never rotate the key.\"}\n{\"id\": \"K119\", \"a\": \"The password must have exactly 8 characters.\", \"b\": \"The password must have at least 8 characters.\"}\n{\"id\": \"K120\", \"a\": \"No, borra todo.\", \"b\": \"Borra todo.\"}\n{\"id\": \"K121\", \"a\": \"Apaga los archivos que nunca se abrieron.\", \"b\": \"Nunca apagues los archivos que se abrieron.\"}\n{\"id\": \"K122\", \"a\": \"Is the port open? No | at most two hosts.\", \"b\": \"Is the port open? No at most two hosts.\"}\n{\"id\": \"K123\", \"a\": \"Never save, but delete.\", \"b\": \"Never save, but delete.\"}\n{\"id\": \"K124\", \"a\": \"Restart all servers except the test server.\", \"b\": \"Restart all servers other than the test server.\"}\n{\"id\": \"K125\", \"a\": \"Restart the server only if it hangs.\", \"b\": \"Restart the server if it hangs.\"}\n{\"id\": \"K126\", \"a\": \"Do not delete the logs, but archive the reports.\", \"b\": \"Do not delete the logs; instead, archive the reports.\"}\n{\"id\": \"K127\", \"a\": \"No apruebes a menos que firmen ambos gerentes.\", \"b\": \"No apruebes a menos que firme uno de los gerentes.\"}\n{\"id\": \"K128\", \"a\": \"Save the file and do not delete the copy.\", \"b\": \"Save the file, and do not delete the copy.\"}\n{\"id\": \"K129\", \"a\": \"Keep logs? No | fewer than five days.\", \"b\": \"Keep logs? No fewer than five days.\"}\n{\"id\": \"K130\", \"a\": \"Keep logs? No, fewer than five days.\", \"b\": \"Keep logs? No/ fewer than five days.\"}\n{\"id\": \"K131\", \"a\": \"Env\u00eda el informe a todos excepto a la directora.\", \"b\": \"Env\u00eda el informe a todos excepto a las directoras.\"}\n{\"id\": \"K132\", \"a\": \"Restart the servers that are not responding.\", \"b\": \"Restart the servers which are not responding.\"}\n{\"id\": \"K133\", \"a\": \"No, como m\u00e1ximo tres.\", \"b\": \"No, m\u00e1ximo tres.\"}\n{\"id\": \"K134\", \"a\": \"Raise the price by 10%.\", \"b\": \"Raise the price by 20%.\"}\n{\"id\": \"K135\", \"a\": \"Never save the file, or delete the copy.\", \"b\": \"Save the file, or never delete the copy.\"}\n{\"id\": \"K136\", \"a\": \"Is it ready? No, not yet.\", \"b\": \"Is it ready? No not yet.\"}\n{\"id\": \"K137\", \"a\": \"No ping the client y quarantine el host.\", \"b\": \"Ping the client y no quarantine el host.\"}\n{\"id\": \"K138\", \"a\": \"Delete every row except the one with the mouse.\", \"b\": \"Delete every row except the ones with the mice.\"}\n{\"id\": \"K139\", \"a\": \"Should I shred the papers? Don't, shred the disk.\", \"b\": \"Should I shred the papers? Don't shred the disk.\"}\n{\"id\": \"K140\", \"a\": \"Reinicia todo, salvo el mes.\", \"b\": \"Reinicia todo, salvo los meses.\"}\n{\"id\": \"K141\", \"a\": \"No pagues la factura, env\u00eda el recibo, llama al banco.\", \"b\": \"Paga la factura, env\u00eda el recibo, no llames al banco.\"}\n{\"id\": \"K142\", \"a\": \"Nunca compartas la clave.\", \"b\": \"Rara vez compartas la clave.\"}\n{\"id\": \"K143\", \"a\": \"Keep logs? No, fewer than five days.\", \"b\": \"Keep logs? No-fewer than five days.\"}\n{\"id\": \"K144\", \"a\": \"Respuesta: **No**\\nMenos de cinco d\u00edas.\", \"b\": \"Respuesta: No menos de cinco d\u00edas.\"}\n{\"id\": \"K145\", \"a\": \"Raise the price by 10%.\", \"b\": \"Lower the price by 10%.\"}\n{\"id\": \"K146\", \"a\": \"You may restart the server.\", \"b\": \"You are allowed to restart the server.\"}\n{\"id\": \"K147\", \"a\": \"Reject the requests that do not include a signature.\", \"b\": \"Reject all requests that do not include a signature.\"}\n{\"id\": \"K148\", \"a\": \"Ningu\u00e9m foi convidado e todos vieram.\", \"b\": \"Todos foram convidados e ningu\u00e9m veio.\"}\n{\"id\": \"K149\", \"a\": \"Delete the file? \\\"No.\\\"\", \"b\": \"Delete the file? No.\"}\n{\"id\": \"K150\", \"a\": \"Keep logs? No \u2013 fewer than five days.\", \"b\": \"Keep logs? No fewer than five days.\"}\n{\"id\": \"K151\", \"a\": \"The vendor that did not deliver must be paid.\", \"b\": \"The vendor that delivered must not be paid.\"}\n{\"id\": \"K152\", \"a\": \"Escalate the cases where no response arrived.\", \"b\": \"Escalate the cases where a response arrived.\"}\n{\"id\": \"K153\", \"a\": \"None of the tests failed.\", \"b\": \"Not all of the tests failed.\"}\n{\"id\": \"K154\", \"a\": \"No, delete everything.\", \"b\": \"Delete everything.\"}\n{\"id\": \"K155\", \"a\": \"Debes hacer la copia.\", \"b\": \"Puedes hacer la copia.\"}\n{\"id\": \"K156\", \"a\": \"Reset every account except the manager's.\", \"b\": \"Reset every account except the managers'.\"}\n{\"id\": \"K157\", \"a\": \"Close every ticket except the blocked ticket.\", \"b\": \"Close every ticket, except the blocked ticket.\"}\n{\"id\": \"K158\", \"a\": \"El usuario est\u00e1 obligado a iniciar sesi\u00f3n.\", \"b\": \"El usuario tiene permitido iniciar sesi\u00f3n.\"}\n{\"id\": \"K159\", \"a\": \"N\u00e3o apague o arquivo: salve a c\u00f3pia.\", \"b\": \"N\u00e3o apague o arquivo; salve a c\u00f3pia.\"}\n{\"id\": \"K160\", \"a\": \"Guarde menos de 5 c\u00f3pias.\", \"b\": \"Guarde no m\u00e1ximo 5 c\u00f3pias.\"}\n{\"id\": \"K161\", \"a\": \"Restart all servers except this server.\", \"b\": \"Restart all servers except these servers.\"}\n{\"id\": \"K162\", \"a\": \"Stop the service before updating it.\", \"b\": \"Stop the service after updating it.\"}\n{\"id\": \"K163\", \"a\": \"No hagas m\u00e1s, pero borra la copia.\", \"b\": \"Haz m\u00e1s, pero no borres la copia.\"}\n{\"id\": \"K164\", \"a\": \"Reserve two rooms for three nights.\", \"b\": \"Reserve three rooms for two nights.\"}\n{\"id\": \"K165\", \"a\": \"Reinicia todo, fuera de la de prueba.\", \"b\": \"Reinicia todo, fuera de las de prueba.\"}\n{\"id\": \"K166\", \"a\": \"Aprueba si A y B.\", \"b\": \"Aprueba si A o B.\"}\n{\"id\": \"K167\", \"a\": \"N\u00e3o envie o relat\u00f3rio ao cliente, mas envie a fatura.\", \"b\": \"Envie o relat\u00f3rio ao cliente, mas n\u00e3o envie a fatura.\"}\n{\"id\": \"K168\", \"a\": \"A n\u00e3o ser que seja urgente, feche o chamado e avise o respons\u00e1vel.\", \"b\": \"Feche o chamado e, a n\u00e3o ser que seja urgente, avise o respons\u00e1vel.\"}\n{\"id\": \"K169\", \"a\": \"Do not restart the server although the patch is ready.\", \"b\": \"Although the patch is ready, do not restart the server.\"}\n{\"id\": \"K170\", \"a\": \"No, do not save the file.\", \"b\": \"No, don't save the file.\"}\n{\"id\": \"K171\", \"a\": \"Open the ticket but never close it.\", \"b\": \"Open the ticket, but never close it.\"}\n{\"id\": \"K172\", \"a\": \"El usuario puede borrar el archivo.\", \"b\": \"El usuario debe borrar el archivo.\"}\n{\"id\": \"K173\", \"a\": \"Restart every server except the ones in staging.\", \"b\": \"Restart every server apart from the ones in staging.\"}\n{\"id\": \"K174\", \"a\": \"Charge $100.\", \"b\": \"Charge \u20ac100.\"}\n{\"id\": \"K175\", \"a\": \"No m\u00e1s de diez.\", \"b\": \"Como m\u00e1ximo diez.\"}\n{\"id\": \"K176\", \"a\": \"Exceto Ana, avise a todos do atraso e do custo.\", \"b\": \"Avise a todos do atraso e, exceto a Ana, do custo.\"}\n{\"id\": \"K177\", \"a\": \"Reinicia todo excepto los servidores de prueba.\", \"b\": \"Reinicia todo a excepci\u00f3n de los servidores de prueba.\"}\n{\"id\": \"K178\", \"a\": \"Guardamos los logs? De ninguna manera, menos de cinco dias.\", \"b\": \"Guardamos los logs? De ninguna manera menos de cinco dias.\"}\n{\"id\": \"K179\", \"a\": \"Envie tudo menos a fatura.\", \"b\": \"Envie tudo menos as faturas.\"}\n{\"id\": \"K180\", \"a\": \"Keep logs? No! Fewer than five days.\", \"b\": \"Keep logs? No fewer than five days.\"}\n{\"id\": \"K181\", \"a\": \"Borra todos los archivos salvo el informe.\", \"b\": \"Borra todos los archivos salvo los informes.\"}\n{\"id\": \"K182\", \"a\": \"Nunca, bajo ninguna circunstancia, desactives la alarma.\", \"b\": \"Nunca bajo ninguna circunstancia desactives la alarma.\"}\n{\"id\": \"K183\", \"a\": \"Deploy unless the tests fail.\", \"b\": \"Deploy unless the tests pass.\"}\n{\"id\": \"K184\", \"a\": \"No quiero m\u00e1s logs.\", \"b\": \"M\u00e1s logs no quiero.\"}\n{\"id\": \"K185\", \"a\": \"Keep logs? No, fewer than five days.\", \"b\": \"Keep logs? No \u2026 fewer than five days.\"}\n{\"id\": \"K186\", \"a\": \"Tell them that the server is not down.\", \"b\": \"Do not tell them that the server is down.\"}\n{\"id\": \"K187\", \"a\": \"Restart everything except the test server.\", \"b\": \"Restart everything but the test server.\"}\n{\"id\": \"K188\", \"a\": \"Cifra el archivo y luego compr\u00edmelo.\", \"b\": \"Comprime el archivo y luego c\u00edfralo.\"}\n{\"id\": \"K189\", \"a\": \"Reinicia todos los servidores, excepto los de prueba.\", \"b\": \"Reinicia todos los servidores, con excepci\u00f3n de los de prueba.\"}\n{\"id\": \"K190\", \"a\": \"Delete all files except the report.\", \"b\": \"Delete all files except a report.\"}\n{\"id\": \"K191\", \"a\": \"Aplica el parche a todos los servidores excepto este.\", \"b\": \"Aplica el parche a todos los servidores excepto estos.\"}\n{\"id\": \"K192\", \"a\": \"Do not, however, delete the file.\", \"b\": \"Do not, though, delete the file.\"}\n{\"id\": \"K193\", \"a\": \"Salve nada, mas apague a c\u00f3pia.\", \"b\": \"Salve o arquivo, mas n\u00e3o apague nada.\"}\n{\"id\": \"K194\", \"a\": \"Rechaza las solicitudes que no incluyen una firma.\", \"b\": \"Rechaza las solicitudes sin firma.\"}\n{\"id\": \"K195\", \"a\": \"N\u00e3o apague os registros, arquive os relat\u00f3rios.\", \"b\": \"Apague os registros, n\u00e3o arquive os relat\u00f3rios.\"}\n{\"id\": \"K196\", \"a\": \"No. Menos de cinco d\u00edas.\", \"b\": \"No menos de cinco d\u00edas.\"}\n{\"id\": \"K197\", \"a\": \"Archive it, but do not delete it.\", \"b\": \"Do not delete it, but archive it.\"}\n{\"id\": \"K198\", \"a\": \"Do not delete the file unless you are sure.\", \"b\": \"Delete the file unless you are sure.\"}\n{\"id\": \"K199\", \"a\": \"Adicione o usu\u00e1rio ao grupo.\", \"b\": \"Remova o usu\u00e1rio do grupo.\"}\n{\"id\": \"K200\", \"a\": \"Never merge the branch, close the ticket; notify QA.\", \"b\": \"Merge the branch, never close the ticket; notify QA.\"}\n{\"id\": \"K201\", \"a\": \"The system must encrypt data.\", \"b\": \"The system should encrypt data.\"}\n{\"id\": \"K202\", \"a\": \"Nobody saves the file, but delete the copy.\", \"b\": \"Save the file, but nobody deletes the copy.\"}\n{\"id\": \"K203\", \"a\": \"Ana says Luis approved it.\", \"b\": \"Luis says Ana approved it.\"}\n{\"id\": \"K204\", \"a\": \"Puede borrar? Nada, mas de cinco archivos.\", \"b\": \"Puede borrar? Nada mas de cinco archivos.\"}\n{\"id\": \"K205\", \"a\": \"Haz copia diaria excepto el domingo.\", \"b\": \"Haz copia diaria excepto los domingos.\"}\n{\"id\": \"K206\", \"a\": \"Todos recebem b\u00f4nus a n\u00e3o ser o novo cliente.\", \"b\": \"Todos recebem b\u00f4nus a n\u00e3o ser os novos clientes.\"}\n{\"id\": \"K207\", \"a\": \"No borres el archivo.\", \"b\": \"N\u00e3o apague o arquivo.\"}\n{\"id\": \"K208\", \"a\": \"Guarda menos de 5 copias.\", \"b\": \"Guarda como m\u00e1ximo 5 copias.\"}\n{\"id\": \"K209\", \"a\": \"Remove the account besides the admin accounts.\", \"b\": \"Remove the accounts besides the admin account.\"}\n{\"id\": \"K210\", \"a\": \"Close the ticket, unless it is urgent, and notify the owner.\", \"b\": \"Close the ticket and notify the owner, unless it is urgent.\"}\n{\"id\": \"K211\", \"a\": \"Restart everything but the test.\", \"b\": \"Restart everything but the tests.\"}\n{\"id\": \"K212\", \"a\": \"Keep at most 5 backups.\", \"b\": \"Keep no more than 5 backups.\"}\n{\"id\": \"K213\", \"a\": \"Update all clients except the one in Madrid and the one in Lima.\", \"b\": \"Update all clients except the one in Madrid and the ones in Lima.\"}\n{\"id\": \"K214\", \"a\": \"Notify everyone except the interns and the contractors.\", \"b\": \"Notify everyone except the interns, and the contractors.\"}\n{\"id\": \"K215\", \"a\": \"Keep no fewer than five servers.\", \"b\": \"Keep at least five servers.\"}\n{\"id\": \"K216\", \"a\": \"Reject every request but the first; approve the last.\", \"b\": \"Reject the first request; approve every request but the last.\"}\n{\"id\": \"K217\", \"a\": \"Should we keep the logs? Nope / fewer than five days.\", \"b\": \"Should we keep the logs? Nope fewer than five days.\"}\n{\"id\": \"K218\", \"a\": \"Keep logs? [No] \u2014 fewer than five days.\", \"b\": \"Keep logs? [No] fewer than five days.\"}\n{\"id\": \"K219\", \"a\": \"What do I owe? Nada \u2192 menos de cinco euros.\", \"b\": \"What do I owe? Nada menos de cinco euros.\"}\n{\"id\": \"K220\", \"a\": \"Keep logs? No (fewer than five days).\", \"b\": \"Keep logs? No fewer than five days.\"}\n{\"id\": \"K221\", \"a\": \"The batch job is not stopped, instead the queue is drained.\", \"b\": \"The batch job is stopped, instead the queue is not drained.\"}\n{\"id\": \"K222\", \"a\": \"Reinicia los servidores que no responden.\", \"b\": \"Reinicia los servidores que no respondan.\"}\n{\"id\": \"K223\", \"a\": \"Elimina los archivos que no est\u00e1n cifrados.\", \"b\": \"Elimina los archivos que est\u00e1n cifrados.\"}\n{\"id\": \"K224\", \"a\": \"No fewer than five days.\", \"b\": \"At least five days.\"}\n{\"id\": \"K225\", \"a\": \"Reinicia todo, excepto el servidor.\", \"b\": \"Reinicia todo, salvo el servidor.\"}\n{\"id\": \"K226\", \"a\": \"Not, strictly speaking, the staging servers.\", \"b\": \"Not strictly speaking the staging servers.\"}\n{\"id\": \"K227\", \"a\": \"Close the ports other than 443.\", \"b\": \"Close the ports other than 443 and 80.\"}\n{\"id\": \"K228\", \"a\": \"Tente novamente a cada 2 horas.\", \"b\": \"Tente novamente a cada 3 horas.\"}\n{\"id\": \"K229\", \"a\": \"Save the file, but nobody should delete the copy.\", \"b\": \"Nobody should delete the copy, but save the file.\"}\n{\"id\": \"K230\", \"a\": \"Restart the container barring the sidecar containers.\", \"b\": \"Restart the containers barring the sidecar container.\"}\n{\"id\": \"K231\", \"a\": \"Never delete the file and never rename it.\", \"b\": \"Never delete the file, and never rename it.\"}\n{\"id\": \"K232\", \"a\": \"Step 1: validate. Step 2: save.\", \"b\": \"Step 1: save. Step 2: validate.\"}\n{\"id\": \"K233\", \"a\": \"Is it ten users? It is not, at least twenty.\", \"b\": \"Is it ten users? It is not at least twenty.\"}\n{\"id\": \"K234\", \"a\": \"Delete the file? No - never.\", \"b\": \"Delete the file? No \u2013 never.\"}\n{\"id\": \"K235\", \"a\": \"N\u00e3o, menos de cinco dias.\", \"b\": \"N\u00e3o. Menos de cinco dias.\"}\n{\"id\": \"K236\", \"a\": \"Reinicie todos os servidores, exceto um de teste.\", \"b\": \"Reinicie todos os servidores, exceto uns de teste.\"}\n{\"id\": \"K237\", \"a\": \"Delete all files except the report.\", \"b\": \"Delete all files with the exception of the report.\"}\n{\"id\": \"K238\", \"a\": \"Fa\u00e7a backup todo dia, exceto o domingo.\", \"b\": \"Fa\u00e7a backup todo dia, exceto os domingos.\"}\n{\"id\": \"K239\", \"a\": \"Users must not delete or edit records.\", \"b\": \"Users may delete, but must not edit records.\"}\n{\"id\": \"K240\", \"a\": \"Delete all files except the report.\", \"b\": \"Delete every file except the report.\"}\n{\"id\": \"K241\", \"a\": \"Keep logs? N0, fewer than five days.\", \"b\": \"Keep logs? N0 fewer than five days.\"}\n{\"id\": \"K242\", \"a\": \"No, more than ten.\", \"b\": \"No, over ten.\"}\n{\"id\": \"K243\", \"a\": \"N\u00e3o ligue a menos que haja uma emerg\u00eancia.\", \"b\": \"Ligue a menos que haja uma emerg\u00eancia.\"}\n{\"id\": \"K244\", \"a\": \"Dime qu\u00e9 tareas no est\u00e1n terminadas.\", \"b\": \"No me digas qu\u00e9 tareas est\u00e1n terminadas.\"}\n{\"id\": \"K245\", \"a\": \"Espera 15 minutos.\", \"b\": \"Espera 15 horas.\"}\n{\"id\": \"K246\", \"a\": \"No borres nada excepto los registros; no toques las copias.\", \"b\": \"No borres nada; no toques las copias excepto los registros.\"}\n{\"id\": \"K247\", \"a\": \"Posso guardar? N\u00e3o \u2192 menos de cinco dias.\", \"b\": \"Posso guardar? N\u00e3o menos de cinco dias.\"}\n{\"id\": \"K248\", \"a\": \"Close the ticket short of the escalated tickets.\", \"b\": \"Close the tickets short of the escalated ticket.\"}\n{\"id\": \"K249\", \"a\": \"Keep logs? \\\"No,\\\" fewer than five days.\", \"b\": \"Keep logs? \\\"No\\\", fewer than five days.\"}\n{\"id\": \"K250\", \"a\": \"Restart everything, except for the test.\", \"b\": \"Restart everything, except for the tests.\"}\n{\"id\": \"K251\", \"a\": \"The server must not be rebooted, because the cache must be flushed.\", \"b\": \"The server must be rebooted, because the cache must not be flushed.\"}\n{\"id\": \"K252\", \"a\": \"Keep at most 5 backups.\", \"b\": \"Keep up to 5 backups.\"}\n{\"id\": \"K253\", \"a\": \"Never save the file, although delete the copy, but archive it.\", \"b\": \"Save the file, although never delete the copy, but archive it.\"}\n{\"id\": \"K254\", \"a\": \"Restart all servers except production ones.\", \"b\": \"Restart all servers except production one.\"}\n{\"id\": \"K255\", \"a\": \"Can I reboot? You can't, more than five users.\", \"b\": \"Can I reboot? You cannot, more than five users.\"}\n{\"id\": \"K256\", \"a\": \"The file was deleted.\", \"b\": \"The file will be deleted.\"}\n{\"id\": \"K257\", \"a\": \"Is the cap ten users? Nobody, at least twenty.\", \"b\": \"Is the cap ten users? Nobody at least twenty.\"}\n{\"id\": \"K258\", \"a\": \"Restart all servers except the ones in test.\", \"b\": \"Restart all servers other than the ones in test.\"}\n{\"id\": \"K259\", \"a\": \"Delete the files that are not encrypted.\", \"b\": \"Delete the files that are unencrypted.\"}\n{\"id\": \"K260\", \"a\": \"Reinicia todos los servidores excepto los servidores de prueba.\", \"b\": \"Reinicia todos los servidores salvo los servidores de prueba.\"}\n{\"id\": \"K261\", \"a\": \"Never save the file, but delete the copy.\", \"b\": \"Never save the file but delete the copy.\"}\n{\"id\": \"K262\", \"a\": \"Every server must restart.\", \"b\": \"All servers must restart.\"}\n{\"id\": \"K263\", \"a\": \"S\u00ed, como m\u00e1ximo tres.\", \"b\": \"No, como m\u00e1ximo tres.\"}\n{\"id\": \"K264\", \"a\": \"Nenhum dos servidores antigos ser\u00e1 reiniciado.\", \"b\": \"Nenhum dos servidores novos ser\u00e1 reiniciado.\"}\n{\"id\": \"K265\", \"a\": \"Reinicie tudo, exceto o resto.\", \"b\": \"Reinicie tudo, exceto os restos.\"}\n{\"id\": \"K266\", \"a\": \"Cierra todas las sesiones excepto la del administrador.\", \"b\": \"Cierra todas las sesiones excepto las del administrador.\"}\n{\"id\": \"K267\", \"a\": \"Notify the client and the manager.\", \"b\": \"Notify the client or the manager.\"}\n{\"id\": \"K268\", \"a\": \"Diga que o servidor n\u00e3o caiu.\", \"b\": \"N\u00e3o diga que o servidor caiu.\"}\n{\"id\": \"K269\", \"a\": \"Limit the upload to 5 MB.\", \"b\": \"Limit the upload to 5 GB.\"}\n{\"id\": \"K270\", \"a\": \"No hay que aplicar el parche, ya que el servicio se reinicia.\", \"b\": \"Hay que aplicar el parche, ya que el servicio no se reinicia.\"}\n{\"id\": \"K271\", \"a\": \"Do not save the file but delete the copy.\", \"b\": \"Don't save the file but delete the copy.\"}\n{\"id\": \"K272\", \"a\": \"If the disk is full, do not save the file.\", \"b\": \"If the disk is full, don't save the file.\"}\n{\"id\": \"K273\", \"a\": \"Encrypt the finance volume, the rest stays open.\", \"b\": \"Encrypt the finance volumes, the rest stays open.\"}\n{\"id\": \"K274\", \"a\": \"Notifica a todos excepto al gerente.\", \"b\": \"Notifica a todos salvo al gerente.\"}\n{\"id\": \"K275\", \"a\": \"Back up the data, but do not restart the server.\", \"b\": \"Back up the data; do not restart the server.\"}\n{\"id\": \"K276\", \"a\": \"Users who did not log in will not receive the email.\", \"b\": \"Users who did not log in will not get the email.\"}\n{\"id\": \"K277\", \"a\": \"Sim, pelo menos dois.\", \"b\": \"Sim, dois ou mais.\"}\n{\"id\": \"K278\", \"a\": \"Keep logs? No // fewer than five days.\", \"b\": \"Keep logs? No fewer than five days.\"}\n{\"id\": \"K279\", \"a\": \"Notify the user who did not sign the form.\", \"b\": \"Do not notify the user who signed the form.\"}\n{\"id\": \"K280\", \"a\": \"No (menos de cinco d\u00edas).\", \"b\": \"No menos de cinco d\u00edas.\"}\n{\"id\": \"K281\", \"a\": \"Delete accounts that never logged in.\", \"b\": \"Never delete accounts that logged in.\"}\n{\"id\": \"K282\", \"a\": \"No, do not delete anything.\", \"b\": \"No, do not delete anything.  \"}\n{\"id\": \"K283\", \"a\": \"Keep logs? No \u2026 fewer than five days.\", \"b\": \"Keep logs? No fewer than five days.\"}\n{\"id\": \"K284\", \"a\": \"Confirm that the backup did not fail.\", \"b\": \"Confirm that the backup has not failed.\"}\n{\"id\": \"K285\", \"a\": \"Env\u00eda todo menos el contrato.\", \"b\": \"Env\u00eda todo salvo el contrato.\"}\n{\"id\": \"K286\", \"a\": \"N\u00e3o salve, mas apague.\", \"b\": \"N\u00e3o salve, por\u00e9m apague.\"}\n{\"id\": \"K287\", \"a\": \"No env\u00edes el informe al cliente pero env\u00eda la factura.\", \"b\": \"Env\u00eda el informe al cliente pero no env\u00edes la factura.\"}\n{\"id\": \"K288\", \"a\": \"Never throttle the API and escalate to oncall unless approved.\", \"b\": \"Throttle the API and never escalate to oncall unless approved.\"}\n{\"id\": \"K289\", \"a\": \"Diga que a fatura n\u00e3o foi paga.\", \"b\": \"N\u00e3o diga que a fatura foi paga.\"}\n{\"id\": \"K290\", \"a\": \"Keep logs? No \u2022 more than five days.\", \"b\": \"Keep logs? No more than five days.\"}\n{\"id\": \"K291\", \"a\": \"\u00bfBorro? No. M\u00e1s de dos archivos.\", \"b\": \"\u00bfBorro? No m\u00e1s de dos archivos.\"}\n{\"id\": \"K292\", \"a\": \"Do not, please, delete the file.\", \"b\": \"Do not please delete the file.\"}\n{\"id\": \"K293\", \"a\": \"No\\n\\nM\u00e1s de diez.\", \"b\": \"No m\u00e1s de diez.\"}\n{\"id\": \"K294\", \"a\": \"No borres la copia, sino guarda el archivo.\", \"b\": \"Guarda el archivo, pero no borres la copia.\"}\n{\"id\": \"K295\", \"a\": \"Keep logs? No * fewer than five days.\", \"b\": \"Keep logs? No fewer than five days.\"}\n{\"id\": \"K296\", \"a\": \"Report the service where no errors occurred.\", \"b\": \"Report the service in which no errors occurred.\"}\n{\"id\": \"K297\", \"a\": \"Do not save the file, but delete the copy.\", \"b\": \"Do not save the file; but delete the copy.\"}\n{\"id\": \"K298\", \"a\": \"Restart every server except the ones in staging.\", \"b\": \"Restart every server, excluding the ones in staging.\"}\n{\"id\": \"K299\", \"a\": \"Rejeite as solicita\u00e7\u00f5es que n\u00e3o incluem assinatura.\", \"b\": \"Rejeite as solicita\u00e7\u00f5es que n\u00e3o incluam assinatura.\"}\n{\"id\": \"K300\", \"a\": \"Every server must restart.\", \"b\": \"At least one server must restart.\"}\n{\"id\": \"K301\", \"a\": \"Keep logs? __No__ - fewer than five days.\", \"b\": \"Keep logs? __No__ fewer than five days.\"}\n{\"id\": \"K302\", \"a\": \"Nunca, jamas, desactives la alarma.\", \"b\": \"Nunca jamas desactives la alarma.\"}\n{\"id\": \"K303\", \"a\": \"Reboot every node barring the primary.\", \"b\": \"Reboot every node barring the primaries.\"}\n{\"id\": \"K304\", \"a\": \"Start the job.\", \"b\": \"Stop the job.\"}\n{\"id\": \"K305\", \"a\": \"Remove all devices save the tablet.\", \"b\": \"Remove all devices save the tablets.\"}\n{\"id\": \"K306\", \"a\": \"Open from Monday to Friday.\", \"b\": \"Open from Monday to Saturday.\"}\n{\"id\": \"K307\", \"a\": \"Should I patch the host? Does not, patch the image.\", \"b\": \"Should I patch the host? Does not patch the image.\"}\n{\"id\": \"K308\", \"a\": \"Keep logs? No\u2026, fewer than five days.\", \"b\": \"Keep logs? No, fewer than five days.\"}\n{\"id\": \"K309\", \"a\": \"Nunca salve o arquivo enquanto apaga a c\u00f3pia.\", \"b\": \"Salve o arquivo enquanto nunca apaga a c\u00f3pia.\"}\n{\"id\": \"K310\", \"a\": \"Archiva los tickets que nadie ha respondido.\", \"b\": \"Archiva los tickets que todos han respondido.\"}\n{\"id\": \"K311\", \"a\": \"Never save the file; delete the copy.\", \"b\": \"Never save the file. Delete the copy.\"}\n{\"id\": \"K312\", \"a\": \"Keep logs? No + fewer than five days.\", \"b\": \"Keep logs? No fewer than five days.\"}\n{\"id\": \"K313\", \"a\": \"Back up the volume not including the scratch volumes.\", \"b\": \"Back up the volumes not including the scratch volume.\"}\n{\"id\": \"K314\", \"a\": \"Rejeite as solicita\u00e7\u00f5es que n\u00e3o incluem assinatura.\", \"b\": \"Rejeite as solicita\u00e7\u00f5es que incluem assinatura.\"}\n{\"id\": \"K315\", \"a\": \"No pagues la factura, env\u00eda el recibo, llama al banco.\", \"b\": \"Paga la factura, no env\u00edes el recibo, llama al banco.\"}\n{\"id\": \"K316\", \"a\": \"Reset every profile except the person in charge.\", \"b\": \"Reset every profile except the people in charge.\"}\n{\"id\": \"K317\", \"a\": \"No, at most 3.\", \"b\": \"Yes, at most 3.\"}\n{\"id\": \"K318\", \"a\": \"Restart all servers except the test servers.\", \"b\": \"Restart all the servers except the test servers.\"}\n{\"id\": \"K319\", \"a\": \"Voc\u00ea pode reiniciar o servidor.\", \"b\": \"Voc\u00ea tem permiss\u00e3o para reiniciar o servidor.\"}\n{\"id\": \"K320\", \"a\": \"Restart all servers except it.\", \"b\": \"Restart all servers except them.\"}\n{\"id\": \"K321\", \"a\": \"Llama a los clientes que no han pagado.\", \"b\": \"Llama a aquellos clientes que no han pagado.\"}\n{\"id\": \"K322\", \"a\": \"Keep at least 5 backups.\", \"b\": \"Keep at least five backups.\"}\n{\"id\": \"K323\", \"a\": \"Permita o acesso se o usu\u00e1rio fez login.\", \"b\": \"Permita o acesso somente se o usu\u00e1rio fez login.\"}\n{\"id\": \"K324\", \"a\": \"Who may approve? No one, at most two managers.\", \"b\": \"Who may approve? No one at most two managers.\"}\n{\"id\": \"K325\", \"a\": \"When the alert fires, don't page the team and open a ticket.\", \"b\": \"When the alert fires, page the team and don't open a ticket.\"}\n{\"id\": \"K326\", \"a\": \"Retry up to 3 times.\", \"b\": \"Retry up to 4 times.\"}\n{\"id\": \"K327\", \"a\": \"Close the tickets that have no owner.\", \"b\": \"Close the tickets which have no owner.\"}\n{\"id\": \"K328\", \"a\": \"Fa\u00e7a backup de todas as pastas exceto as que t\u00eam arquivos tempor\u00e1rios.\", \"b\": \"Fa\u00e7a backup de todas as pastas exceto as que n\u00e3o t\u00eam arquivos tempor\u00e1rios.\"}\n{\"id\": \"K329\", \"a\": \"Apply the patch to all servers except this one.\", \"b\": \"Apply the patch to all servers except these ones.\"}\n{\"id\": \"K330\", \"a\": \"Do X after Y.\", \"b\": \"Do X before Y.\"}\n{\"id\": \"K331\", \"a\": \"Reinicia todo excepto el de A, el de B y el de C.\", \"b\": \"Reinicia todo excepto el de A, el de B y los de C.\"}\n{\"id\": \"K332\", \"a\": \"Apague os arquivos que n\u00e3o est\u00e3o criptografados.\", \"b\": \"Apague todos os arquivos que n\u00e3o est\u00e3o criptografados.\"}\n{\"id\": \"K333\", \"a\": \"Revise todos os contratos, a excecao do do cliente.\", \"b\": \"Revise todos os contratos, a excecao dos do cliente.\"}\n{\"id\": \"K334\", \"a\": \"Llama a los clientes que no han pagado.\", \"b\": \"Llama a los clientes que no hayan pagado.\"}\n{\"id\": \"K335\", \"a\": \"Reinicia todos los servidores, salvo tu servidor.\", \"b\": \"Reinicia todos los servidores, salvo tus servidores.\"}\n{\"id\": \"K336\", \"a\": \"Should we page oncall? No \u2192 more than three alerts.\", \"b\": \"Should we page oncall? No more than three alerts.\"}\n{\"id\": \"K337\", \"a\": \"Contact the clients who have not paid.\", \"b\": \"Contact the clients that have not paid.\"}\n{\"id\": \"K338\", \"a\": \"Do not save the file, but delete the copy.\", \"b\": \"Save the file but, do not delete the copy.\"}\n{\"id\": \"K339\", \"a\": \"Please send the report to the client.\", \"b\": \"Send the report to the client.\"}\n{\"id\": \"K340\", \"a\": \"Notify everyone except the manager.\", \"b\": \"Notify everyone except the managers.\"}\n{\"id\": \"K341\", \"a\": \"Back up every table except the audit table.\", \"b\": \"Back up every table apart from the audit table.\"}\n{\"id\": \"K342\", \"a\": \"Reinicie todos os servidores, tirando o de teste.\", \"b\": \"Reinicie todos os servidores, tirando os de teste.\"}\n{\"id\": \"K343\", \"a\": \"Posso reiniciar? Nao, obrigado.\", \"b\": \"Posso reiniciar? Nao obrigado.\"}\n{\"id\": \"K344\", \"a\": \"Env\u00eda todo menos la factura.\", \"b\": \"Env\u00eda todo menos las facturas.\"}\n{\"id\": \"K345\", \"a\": \"No, bajo ninguna circunstancia, borres el log.\", \"b\": \"No bajo ninguna circunstancia borres el log.\"}\n{\"id\": \"K346\", \"a\": \"Do not save the file, but delete the copy.\", \"b\": \"Delete the copy, but don't save the file.\"}\n{\"id\": \"K347\", \"a\": \"Bob: Delete the backup?\\nAmy: No, fewer than five copies.\", \"b\": \"Bob: Delete the backup?\\nAmy: No fewer than five copies.\"}\n{\"id\": \"K348\", \"a\": \"Borra el archivo excepto el informe.\", \"b\": \"Borra el archivo excepto los informes.\"}\n{\"id\": \"K349\", \"a\": \"Reboot the nodes that are not in the cluster which is healthy.\", \"b\": \"Do not reboot the nodes that are in the cluster which is healthy.\"}\n{\"id\": \"K350\", \"a\": \"Paga dentro de 30 d\u00edas.\", \"b\": \"Paga despu\u00e9s de 30 d\u00edas.\"}\n{\"id\": \"K351\", \"a\": \"Embora n\u00e3o seja urgente, envie.\", \"b\": \"Envie, embora n\u00e3o seja urgente.\"}\n{\"id\": \"K352\", \"a\": \"No, menos de cinco d\u00edas.\", \"b\": \"No: menos de cinco d\u00edas.\"}\n{\"id\": \"K353\", \"a\": \"Confirma que la copia no fall\u00f3.\", \"b\": \"Confirma que no fall\u00f3 la copia.\"}\n{\"id\": \"K354\", \"a\": \"Ana deve assinar, mas Lu\u00eds n\u00e3o deve assinar.\", \"b\": \"Ana deve assinar; Lu\u00eds n\u00e3o deve assinar.\"}\n{\"id\": \"K355\", \"a\": \"Keep logs? No ~ fewer than five days.\", \"b\": \"Keep logs? No fewer than five days.\"}\n{\"id\": \"K356\", \"a\": \"Importante: no apagues el equipo y desconecta el cable.\", \"b\": \"Importante: apaga el equipo y no desconectes el cable.\"}\n{\"id\": \"K357\", \"a\": \"Se o disco estiver cheio, pare o trabalho.\", \"b\": \"Se o trabalho parar, o disco est\u00e1 cheio.\"}\n{\"id\": \"K358\", \"a\": \"Reinicia todos los servidores, quitando el de prueba.\", \"b\": \"Reinicia todos los servidores, quitando los de prueba.\"}\n{\"id\": \"K359\", \"a\": \"El sistema debe cifrar los datos.\", \"b\": \"El sistema deber\u00eda cifrar los datos.\"}\n{\"id\": \"K360\", \"a\": \"No debes desplegar el viernes.\", \"b\": \"No despliegues el viernes.\"}\n"
import json
PAIRS = [json.loads(l) for l in DATA.splitlines() if l.strip()]
print(len(PAIRS), 'pairs')

In [ ]:
import re, time, gc, json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
SYSTEM = RULES + "\nIMPORTANT FORMAT OVERRIDE: reply with ONE word only: SAME, DIFFERENT or UNSURE."
MODELS = [("Qwen/Qwen2.5-7B-Instruct", 12), ("tiiuae/Falcon3-7B-Instruct", 12), ("microsoft/Phi-3.5-mini-instruct", 16)]
RESULTS = {}
def run(name, bs):
    tok = AutoTokenizer.from_pretrained(name); tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(name, quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4"), device_map="cuda:0", low_cpu_mem_usage=True)
    model.eval()
    prompts = []
    for p in PAIRS:
        msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": json.dumps({"id": p["id"], "a": p["a"], "b": p["b"]}, ensure_ascii=False)}]
        prompts.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
    order = sorted(range(len(PAIRS)), key=lambda i: len(prompts[i])); out = {}; t0 = time.time()
    for s in range(0, len(order), bs):
        idx = order[s:s+bs]
        enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True).to("cuda:0")
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=5, do_sample=False, pad_token_id=tok.pad_token_id)
        for i, g in zip(idx, gen):
            txt = tok.decode(g[enc["input_ids"].shape[1]:], skip_special_tokens=True).upper()
            m = re.search(r"SAME|DIFFERENT|UNSURE", txt)
            out[PAIRS[i]["id"]] = m.group(0) if m else "INVALID"
    print(name, len(out), "pairs in", round(time.time() - t0), "s")
    del model; gc.collect(); torch.cuda.empty_cache()
    return out
for name, bs in MODELS:
    try:
        RESULTS[name] = run(name, bs)
    except Exception as e:
        RESULTS[name] = {"ERROR": repr(e)[:300]}; print("FAILED", name, repr(e)[:300])
print("DONE")

In [ ]:
import collections
for name, r in RESULTS.items():
    print("### MODEL", name, dict(collections.Counter(r.values())) if "ERROR" not in r else r)
for name, r in RESULTS.items():
    if "ERROR" in r: continue
    print("@@@", name)
    print(";".join(f"{k}={v[0]}" for k, v in sorted(r.items())))
print("@@@END")